In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import sys
import pathlib
import pickle
import abc
import dataclasses
import typing
import functools

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import seaborn as sns
import datasets

sys.path.append("src/")

import core
import expt
import util

DATA_DIR = util.DEFAULT_DATA_PATH
RESULT_DIR = pathlib.Path("data") / "results"

# Eval classes/functions

In [ ]:
import huggingface_hub
import pyarrow.dataset as ds


IMAGE_LIKE_COLUMNS = {
    "image", "augmented_image", "marked_image",
    "depth_map", "targ_seg", "dist_seg"
}

GT_COLUMNS = [
    "image_name", "env_cat", "env_id", "obj_cat", "obj_id",
    "cues" ,"cue_strength", "odd_position",
]


@functools.lru_cache()
def _load_gt_from_huggingface(subset_name):
    dataset_path = f"datasets/liuyiqian/O3-D/{subset_name}/"
    hf_fs = huggingface_hub.HfFileSystem()
    print("loading metadata via pyArrow:", dataset_path)
    arrow_dataset = ds.dataset(dataset_path, filesystem=hf_fs, format="parquet")

    non_img_columns = list(set(arrow_dataset.schema.names)-IMAGE_LIKE_COLUMNS)
    df = arrow_dataset.to_table(columns=non_img_columns).to_pandas()

    if subset_name == "real-012cue-cropped":
        df = (df.assign(orig_image_name=lambda df: df.image_name.str.replace(r"-t[lr](far|near)", r"", regex=True))
                 .drop_duplicates("orig_image_name"))
    return df 

In [ ]:
@dataclasses.dataclass
class EvaluationDefinition:
    evaluator: core.BaseEvaluator
    ResultCls: type
    GT: typing.Union[type, core.BaseGroundTruth]
    gt_src_getter: callable=lambda x: x
    validate_row: callable=None

In [ ]:
def _row_eval_func(row, eval_def: EvaluationDefinition):
    if eval_def.validate_row is not None:
        validated = eval_def.validate_row(row)
        if not validated:
            return None

    result = eval_def.ResultCls.from_record(row.to_dict())

    if eval_def.gt_src_getter is None: # some GT does not need src
        assert isinstance(eval_def.GT, core.BaseGroundTruth)
        gt = eval_def.GT
    else:
        assert isinstance(eval_def.GT, type)
        gt_src = eval_def.gt_src_getter(row)
        gt = eval_def.GT(gt_src)

    try:
        metric = eval_def.evaluator.evaluate(result, gt)
    except Exception as e:
        print(repr(e))
        raise e
    return metric

def run_analysis_pipeline(
    eval_defs: typing.List[EvaluationDefinition],
    result_pickle_path: str,
    filter_res: callable=lambda x: x,  # can be used as proproc as well
    merge_res_gt: callable=lambda x: x,
):
    res = util.load_pickled_to_df(result_pickle_path)
    print("Loaded orig results with shape:", res.shape)
    res = filter_res(res)

    res_ = merge_res_gt(res)
    print("Filtered & Merged with GT df; new shape:", res_.shape)

    for eval_def in eval_defs:
        metric_ser = res_.apply(_row_eval_func, eval_def=eval_def, axis=1)
        if not isinstance(metric_ser, pd.Series):
            raise ValueError(f"Expecting pd series, but got: {metric_ser}")

        metric_col = eval_def.evaluator.__class__.__name__.replace("Evaluator", "")
        res_[metric_col] = metric_ser

    return res_


def show_img(im_set: expt.BaseImageSet, name: str):
    im_path = im_set.get_image_path(name)
    plt.imshow(plt.imread(im_path))
    plt.title(name)

In [ ]:
def plot_corr_heatmap(corr):
    _figsize = (len(corr), len(corr))
    return sns.heatmap(
        corr,
        vmin=-1, center=0, vmax=1,
        annot=True, fmt='.2f',
        square=True, cmap='coolwarm',
        cbar=False,
        mask=np.triu(np.ones_like(corr)),
        annot_kws=dict(fontsize=14),
        ax=plt.subplots(figsize=_figsize)[1])

In [ ]:
close_far_q_filter = lambda df: df.ques_kind == "closer_farther"

hp_cue_filter = lambda df: df.cues.apply(lambda l: "HP" in l)
tg_cue_filter = lambda df: df.cues.apply(lambda l: "TG" in l)
rs_cue_filter = lambda df: df.cues.apply(lambda l: "RS" in l)
ls_cue_filter = lambda df: df.cues.apply(lambda l: "LS" in l)
sa_cue_filter = lambda df: df.cues.apply(lambda l: "SA" in l)
oc_cue_filter = lambda df: df.cues.apply(lambda l: "OC" in l)

non_icl_filter = lambda df: df.icl.fillna(False) == False

### QA parsing

In [ ]:
import re

YN_ANSWER_MAP = {
    ('in front of', 'yes'): 'closer',
    ('in front of', 'no'): 'farther',
    ('behind', 'yes'): 'farther',
    ('behind', 'no'): 'closer',
    ('at the rear of', 'yes'): 'farther',
    ('at the rear of', 'no'): 'closer',
    ('in front of', 'true'): 'closer',
    ('in front of', 'false'): 'farther',
}

ANSWER_REPHRASING_MAP = {
    'closer': 'closer',
    'farther': 'farther',

    'behind': 'farther',
    'before': 'closer',
    'after': 'farther',
    'nearer': 'closer',
    'in front of': 'closer',
}

REGEX_PATT_MCQ = re.compile(r"(?:\?|\.) ([A-Z]\. [^\.]+\. )([A-Z]\. [^\.]+\. )([A-Z]\. [^\.]+\. )?([A-Z]\. [^\.]+\. )?")

def _parse_mc(row):
    if "A. " not in row.ques:
        return row.answer  # orig answer

    # build option_map
    matched = re.search(REGEX_PATT_MCQ, row.ques)
    option_map = {}  # letter -> answer
    for i in range(1, 5):
        option = matched.group(i)
        if option is not None:
            letter = option[0]
            answer = option.split(" ", maxsplit=1)[1].strip(" .")
            option_map[letter.lower()] = answer

    actual_answer = option_map.get(row.answer.lower(), row.answer.lower())
    # print(option_map, row.answer.lower(), actual_answer)

    for (keyword, yn), answer in YN_ANSWER_MAP.items():
        if actual_answer.lower() == yn and keyword in row.ques.lower():
            actual_answer = answer
            break
        
    return actual_answer

def _parse_yn(row):
    if 'Yes' not in row.ques or 'True' not in row.ques:
        return row.answer  # not a YN question

    actual_answer = row.answer  # assuming already yes/no
    for (keyword, yn), answer in YN_ANSWER_MAP.items():
        if actual_answer.lower() == yn and keyword in row.ques.lower():
            actual_answer = answer

    return actual_answer

def _convert_mc_answer(df):
    return df.assign(answer=lambda df: df.apply(_parse_mc, axis=1))

def _convert_yn_answer(df):
    return df.assign(answer=lambda df: df.apply(_parse_yn, axis=1))

def _rephrasing_answer(df):
    """Anything not in the MAP, will be `np.nan`"""
    # def _rephrasing(s):
    #     rephrased = ANSWER_REPHRASING_MAP.get(s, s)
    #     return rephrased
    return df.assign(answer=lambda df: df.answer.str.lower().map(ANSWER_REPHRASING_MAP))

def _fillna_answer(df):
    return df.assign(answer=lambda df: df.answer.fillna("na"))

#### short answer map

In [ ]:
SHORT_ANSWER_MAP = {
    "right side": "right",
    "left side": "left",

    "top-left quadrant": "top-left",
    "top-right quadrant": "top-right",
    "'top-right' quadrant": "top-right",
    "bottom-right quadrant": "bottom-right",
    "'bottom-right' quadrant": "bottom-right",
    "bottom-left quadrant": "bottom-left",

    "closer to the viewer": "closer",
    "closer to the viewpoint": "closer",
    "closer to viewpoint": "closer",
    "closer to the camera": "closer",
    "in front of": "closer",
    "front": "closer",
    "ahead of the others": "closer",
    "closer to you": "closer",
    "closer to viewer": "closer",
    "closer to us": "closer",
    "farther from the viewpoint": 'farther',
    "farther from the observer": 'farther',
    "farther away from the viewer": 'farther',
    "further away from the viewer": 'farther',
    "farther away from the camera": 'farther',
    "farther from the viewer": "farther",
    'furthest from the viewer': 'farther',
    'farther away than': 'farther',
    'is farther away': 'farther',
    "behind": "farther",
    "greater depth": "farther",
    "farther away compared to": "farther",
    'further back': 'farther',
    'appear farther away': 'farther',
    'positioned further': 'farther',
    'positioned farther': 'farther',
    'positioned after': 'farther',
    'positioned before': 'closer',
    'positioned closer to': 'closer',
    'comes before': 'closer',
    'comes after': 'farther',
    'nearer than': 'closer',
    'appears closer than': 'closer',

    "not possible": "unsure",
    "similar depth as": "unsure",
    "have to say 'Unsure'": "unsure",

    # for cambrian
    "correct answer is A": "A",
    "correct answer is B": "B",
    "correct answer is C": "C",
    "correct answer is D": "D",
    # GPT
    "Answer A": "A",
    "Answer B": "B",
    "Answer: A": "A",
    "Answer: B": "B",
}

COT_PREFIXS = [
    "(Let me think step by step",
    "(Let's think step by step",
]

LEADING_STRINGS = [  # needs fixed-length
    r"Answer: ",
    r"wer is: ",
    r"swer is ",
    r"d say:\n\n",
    r"er is:\n\n",  # answer is:\n\n
    r"ons is: ",  # options is:
    r"itioned ",  # positioned B. Farther
    r"\w comes ", # comes B. After
]
REGEX_LEADING_ANSWER = "|".join(LEADING_STRINGS)
REGEX_PATT_SHORT_ANSWER = re.compile(rf"(?<={REGEX_LEADING_ANSWER})([ABCD])(?=\.|$)")
REGEX_PATT_SHORT_ANSWER_LINE_END = re.compile(r" ([ABCD])\. \w+\.$")
SHORT_ANSWER_PATTERNS = [
    REGEX_PATT_SHORT_ANSWER,
    REGEX_PATT_SHORT_ANSWER_LINE_END,
]

REGEX_PATT_SHORT_ANSWER_MULTILINE = re.compile(r"([ABCD])(?=\.(\w+)?|$)")


def _extract_short_answer(df):
    df_ = df.assign(orig_answer=df.answer)

    def _single_sentence(orig_answer):
        for pre in COT_PREFIXS:
            if pre in orig_answer:  # CoT
                orig_answer = orig_answer.split(")", maxsplit=1)[-1].strip(" ")
                break

        # some regex matching
        for patt in SHORT_ANSWER_PATTERNS:
            re_matched = re.search(patt, orig_answer)
            if re_matched:
                return re_matched.group(1)
        
        if "\n" in orig_answer:
            last_line = orig_answer.split("\n")[-1]  # last line
            re_matched = re.match(REGEX_PATT_SHORT_ANSWER_MULTILINE, last_line)
            if re_matched:
                return re_matched.group(1)
        
        return orig_answer.split(".")[0]  # first sentence

    def _match_phrase(string):
        for phrase, short_answer in SHORT_ANSWER_MAP.items():
            if phrase.lower() in string.lower():
                return short_answer
        return string
    
    df_['answer'] = df_.answer.map(_single_sentence)
    df_['answer'] = df_.answer.map(_match_phrase)
    return df_
    

# answer -> orig_answer
# answer is a new column for parsed answer
_parse_answer = lambda df: (
    df.pipe(_extract_short_answer)
        .pipe(_convert_mc_answer).pipe(_convert_yn_answer)
        .pipe(_rephrasing_answer)
        .pipe(_fillna_answer)
)

#### qtags_map

In [ ]:
def _qtags_map(ques):
    ques_ = ques.lower()
    tags = []
    if "short answer preferred" in ques_:
        tags.append('sap')
    else:
        tags.append('mcq')

    if 'feel' in ques_ or 'come before' in ques_:
        tags.append('lvl-casual')
    elif 'along the line' in ques_ or 'relative to the cam' in ques_:
        tags.append('lvl-formal')
    else:
        tags.append('lvl-avg')

    yn_ques_kws = [
        "yes.",
        "true or false:",
    ] 
    if any(kw in ques_ for kw in yn_ques_kws):
        tags.append('yn')

    reversed_opt_kws = [
        'a. farther.', 
        'a. in front of.', 
        'a. after.',
        'a. no.',
        'a. false.',
    ]
    if any(kw in ques_ for kw in reversed_opt_kws):
        tags.append('ro')

    if 'farther' in ques_:
        tags.append('do-cf')
    elif 'in front of' in ques_ or 'rear' in ques_:
        tags.append('do-rf')
    elif 'before or after' in ques_:
        tags.append('do-ba')
    else:
        pass

    # L/R referring, for real-012cue-cropped
    if "about the right object" in ques_:
        tags.append('35r')
    if "about the left object" in ques_:
        tags.append('35l')
    
    return tags

def _extract_ques_tags(df):
    return df.assign(qtags=df.ques.apply(_qtags_map))

_parse_ques = lambda df: df.pipe(_extract_ques_tags)

_parse_qa = lambda df: df.pipe(_parse_ques).pipe(_parse_answer)

### result-GT merge funcs

In [ ]:
def _merge_res_gt(
    res_preproc: callable,
    gt_subset_name: str,
    gt_col: str,
    res: pd.DataFrame,
):
    res = res_preproc(res.copy())
    # print("res to merge", res.head())
    gt_df = _load_gt_from_huggingface(gt_subset_name)
    # print("gt to merge", gt_df.head())
    if gt_col != "image_name":
        gt_df = gt_df.rename(columns={"image_name": "gt_image_name"})
    res_ = res.merge(gt_df, left_on="image_name", right_on=gt_col)
    return res_

In [ ]:
def _format_orig_image_name(df):
    return (
        df.assign(**{"full_image_name": df.image_name})
            .assign(image_name=lambda df: df.image_name.str.replace(r"-augmented-[a-z-]+", r"", regex=True))
    )

_merge_kb1m_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name,  # from aug to orig
    "kb-1cue",
    "image_name")

_merge_kb2m_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name,  # from aug to orig
    "kb-2cue",
    "image_name")

_merge_kb0m_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name,  # from aug to orig
    "kb-0cue",
    "image_name")

_merge_kb0lpm_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name,  # from aug to orig
    "kb-no-lp",
    "image_name")

In [ ]:
def _format_orig_image_name_o3dcf(df):
    return (
        df.assign(**{"full_image_name": df.image_name})
            .assign(image_name=lambda df: df.image_name.str.replace(r"(-t[lr](far|near))?-augmented-[a-z-]+", r"", regex=True))
    )

_merge_o3dr_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name,  # from aug to orig
    "real-012cue",
    "image_name")

_merge_o3drcf_res_gt = functools.partial(
    _merge_res_gt,
    _format_orig_image_name_o3dcf,
    "real-012cue-cropped",
    "orig_image_name")

_merge_o3do_res_gt = functools.partial(
    _merge_res_gt,
    lambda x: x,
    "real-mcue",
    "image_name",
)

### util for notebook

In [ ]:
def set_mpl_dpi(dpi=150):
    mpl.rcParams['figure.dpi'] = dpi

In [ ]:
def extract_model_name(result_path):
    parent_dirname = pathlib.Path(result_path).parent.name
    return parent_dirname

extract_model_name("data/results/VILA1.5-3b/kubric-fine-mark_depth-order-full-v2_records_20250604192905.pkl").title()

In [ ]:
def extract_is_marked(result_path):
    filename = pathlib.Path(result_path).name
    # print(filename)
    return "-mark" in filename

extract_is_marked("data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-fine-aug_depth-order-full-v2_records_20250605150318.pkl")
extract_is_marked("data/results/google/paligemma2-3b-mix-224/o3d-min-mark-ss_depth-order-full-v2-o3dm_records_20250605154130.pkl")

In [ ]:
aug_filter = lambda df: ~df.is_marked & df.ques_clarity.between(2,3.5)
mark_filter = lambda df: df.is_marked & df.ques_clarity.eq(4)
aug_mark_filter = lambda df: (~df.is_marked & df.ques_clarity.between(2,3.5)) | (df.is_marked & df.ques_clarity.eq(4))

reg_strength_filter = lambda df: df.cue_strength.between(0.9, 1.0)

In [ ]:
def _df_sbg(df, **kwargs):
    vmin = kwargs.pop('vmin', 0)
    vmax = kwargs.pop('vmax', 1)
    cmap = kwargs.pop('cmap', "coolwarm")
    return df.style.background_gradient(vmin=vmin, vmax=vmax, cmap=cmap, **kwargs)

def _df_add_global_info(df):
    return (df.assign(model_name=extract_model_name(path),
                      is_marked=extract_is_marked(path))
                .assign(model_display_name=lambda df: df.model_name.str.lower())
           )

def _df_add_row_info(df):
    return (df.assign(cues=lambda df: df.cues.apply(eval).apply(sorted).apply(tuple),
                      cue_strength_str=lambda df: df.cue_strength.astype(str).str.slice(0, 4),
                      env_id=lambda df: df.env_id.fillna("basic"),
                     )
           )

def _df_combine_metrics(df, prefix) -> pd.Series:
    # assuming one prefix metric for each subframe
    assert df.filter(regex=f"^{prefix}").notna().sum(axis=1).eq(1).all()
    return df.filter(regex=f"^{prefix}").fillna(0).astype(float).sum(axis=1)

# result file paths

## kb-1cue

In [ ]:
!find data/results/ -wholename "*bbox-ref/*" -name "*-1cue*depth-order*"

In [ ]:
SINGLE_CUE_RESULT_PATHS = ("""data/results/llava-v1.5-7b/kubric-1cue-aug_depth-order-high35c-rand_records_20260305131706.pkl
data/results/llava-v1.5-7b/kubric-1cue-aug_depth-order-highc-rand_records_20260304042912.pkl
data/results/llava-v1.5-7b/kubric-1cue-aug_depth-order-medc-rand_records_20260309212748.pkl
data/results/llava-v1.5-7b/kubric-1cue-mark_depth-order-higherc-rand_records_20260304122207.pkl
data/results/google/paligemma2-3b-mix-224/kubric-1cue-aug_depth-order-highc-rand_records_20260304232945.pkl
data/results/google/paligemma2-3b-mix-224/kubric-1cue-aug_depth-order-high35c-rand_records_20260305131607.pkl
data/results/google/paligemma2-3b-mix-224/kubric-1cue-mark_depth-order-higherc-rand_records_20260304075721.pkl
data/results/cambrian-phi3-3b/kubric-1cue-aug_depth-order-highc-rand_records_20260224130954.pkl
data/results/cambrian-phi3-3b/kubric-1cue-aug_depth-order-high35c-rand_records_20260225182115.pkl
data/results/cambrian-phi3-3b/kubric-1cue-mark_depth-order-higherc-rand_records_20260227014020.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-mark_depth-order-higherc-rand_records_20260203143446.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-high35c-rand_records_20260130212436.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-high35c-rand_records_20260129203441.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-highc-rand_records_20260201164320.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-highc-rand_records_20260205210712.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-mark_depth-order-higherc-rand_records_20260212015719.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-medc-rand_records_20260227161644.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-1cue-aug_depth-order-medc-rand_records_20260227160746.pkl
data/results/blip2-flan-t5-xl/kubric-1cue-aug_depth-order-highc-rand_records_20260226130518.pkl
data/results/blip2-flan-t5-xl/kubric-1cue-mark_depth-order-higherc-rand_records_20260226171658.pkl
data/results/blip2-flan-t5-xl/kubric-1cue-aug_depth-order-high35c-rand_records_20260226160421.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-medc-rand_records_20260310181353.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-mark_depth-order-higherc-rand_records_20260302041238.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-highc-rand_records_20260302045834.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-high35c-rand_records_20260302045104.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-mark_depth-order-higherc-rand_records_20260302042531.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-highc-rand_records_20260302045807.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-high35c-rand_records_20260302045057.pkl
data/results/gemini-2.5-flash-lite/kubric-1cue-aug_depth-order-medc-rand_records_20260309211427.pkl
data/results/kosmos-2-patch14-224/kubric-1cue-aug_depth-order-high35c-rand_records_20260226171639.pkl
data/results/kosmos-2-patch14-224/kubric-1cue-aug_depth-order-medc-rand_records_20260309212123.pkl
data/results/kosmos-2-patch14-224/kubric-1cue-aug_depth-order-highc-rand_records_20260226132850.pkl
data/results/kosmos-2-patch14-224/kubric-1cue-mark_depth-order-higherc-rand_records_20260227012909.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-mark_depth-order-higherc-rand_records_20260221012649.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-high35c-rand_records_20260212015606.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-mark_depth-order-higherc-rand_records_20260218103231.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-medc-rand_records_20260312015854.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-high35c-rand_records_20260220135734.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-highc-rand_records_20260220052510.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-medc-rand_records_20260318185455.pkl
data/results/Phi-3.5-vision-instruct/kubric-1cue-aug_depth-order-highc-rand_records_20260208221237.pkl
data/results/gpt-4.1-mini/kubric-1cue-mark_depth-order-higherc-rand_records_20260224010847.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-high35c-rand_records_20260223073440.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-high35c-rand_records_20260214082315.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-medc-rand_records_20260302044138.pkl
data/results/gpt-4.1-mini/kubric-1cue-mark_depth-order-higherc-rand_records_20260215000048.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-medc-rand_records_20260302043809.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-highc-rand_records_20260222111012.pkl
data/results/gpt-4.1-mini/kubric-1cue-aug_depth-order-highc-rand_records_20260213074518.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-high35c-rand_records_20260222125451.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-high35c-rand_records_20260122162007.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-mark_depth-order-higherc-rand_records_20260223004754.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-highc-rand_records_20260129190132.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-mark_depth-order-higherc-rand_records_20260127155157.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-medc-rand_records_20260303132424.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-highc-rand_records_20260222023817.pkl
data/results/DeepSeek-VL-7B-chat/kubric-1cue-aug_depth-order-medc-rand_records_20260227163633.pkl
data/results/InternVL2_5-4B/kubric-1cue-mark_depth-order-higherc-rand_records_20260224153021.pkl
data/results/InternVL2_5-4B/kubric-1cue-aug_depth-order-high35c-rand_records_20260223090000.pkl
data/results/InternVL2_5-4B/kubric-1cue-aug_depth-order-highc-rand_records_20260222013436.pkl
data/results/VILA1.5-3b/kubric-1cue-aug_depth-order-medc-rand_records_20260309212736.pkl
data/results/VILA1.5-3b/kubric-1cue-mark_depth-order-higherc-rand_records_20260225005430.pkl
data/results/VILA1.5-3b/kubric-1cue-aug_depth-order-high35c-rand_records_20260224153332.pkl
data/results/VILA1.5-3b/kubric-1cue-aug_depth-order-highc-rand_records_20260224125935.pkl""").splitlines()
len(SINGLE_CUE_RESULT_PATHS)

## kb-2cue

In [ ]:
!find data/results/ \
    ! -wholename "*fsoc*" ! -wholename "*bbox-ref*" \
    -name "*-2cue*depth-order*"

In [ ]:
CUE_INTERACTION_RESULT_PATHS = ("""data/results/llava-v1.5-7b/kubric-2cue-aug_depth-order-high35c-rand_records_20260305022809.pkl
data/results/llava-v1.5-7b/kubric-2cue-mark_depth-order-higherc-rand_records_20260304163034.pkl
data/results/llava-v1.5-7b/kubric-2cue-aug_depth-order-highc-rand_records_20260304163155.pkl
data/results/llava-v1.5-7b/kubric-2cue-aug_depth-order-medc-rand_records_20260310012702.pkl
data/results/google/paligemma2-3b-mix-224/kubric-2cue-aug_depth-order-high35c-rand_records_20260305131607.pkl
data/results/google/paligemma2-3b-mix-224/kubric-2cue-mark_depth-order-higherc-rand_records_20260304162835.pkl
data/results/google/paligemma2-3b-mix-224/kubric-2cue-aug_depth-order-highc-rand_records_20260305002306.pkl
data/results/cambrian-phi3-3b/kubric-2cue-mark_depth-order-higherc-rand_records_20260303133507.pkl
data/results/cambrian-phi3-3b/kubric-2cue-aug_depth-order-highc-rand_records_20260228021154.pkl
data/results/cambrian-phi3-3b/kubric-2cue-aug_depth-order-high35c-rand_records_20260301205208.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-highc-rand_records_20260212015710.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-mark_depth-order-higherc-rand_records_20260205033114.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-high35c-rand_records_20260219071219.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-highc-rand_records_20260223073831.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-medc-rand_records_20260301091027.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-mark_depth-order-higherc-rand_records_20260214082615.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-medc-rand_records_20260301195134.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-aug_depth-order-high35c-rand_records_20260208160306.pkl
data/results/blip2-flan-t5-xl/kubric-2cue-aug_depth-order-highc-rand_records_20260227013022.pkl
data/results/blip2-flan-t5-xl/kubric-2cue-aug_depth-order-high35c-rand_records_20260227053635.pkl
data/results/blip2-flan-t5-xl/kubric-2cue-mark_depth-order-higherc-rand_records_20260227091157.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-high35c-rand_records_20260302131108.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-highc-rand_records_20260302131751.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-medc-rand_records_20260310035939.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-mark_depth-order-higherc-rand_records_20260302130724.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-medc-rand_records_20260310181409.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-mark_depth-order-higherc-rand_records_20260302130652.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-high35c-rand_records_20260302132538.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-aug_depth-order-highc-rand_records_20260302143023.pkl
data/results/kosmos-2-patch14-224/kubric-2cue-mark_depth-order-higherc-rand_records_20260303235450.pkl
data/results/kosmos-2-patch14-224/kubric-2cue-aug_depth-order-high35c-rand_records_20260227094109.pkl
data/results/kosmos-2-patch14-224/kubric-2cue-aug_depth-order-medc-rand_records_20260309235918.pkl
data/results/kosmos-2-patch14-224/kubric-2cue-aug_depth-order-highc-rand_records_20260227053325.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-medc-rand_records_20260317221704.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-medc-rand_records_20260313172938.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-mark_depth-order-higherc-rand_records_20260218103218.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-high35c-rand_records_20260215125028.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-mark_depth-order-higherc-rand_records_20260222011706.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-highc-rand_records_20260215101452.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-high35c-rand_records_20260221121824.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-aug_depth-order-highc-rand_records_20260221064350.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-medc-rand_records_20260303131755.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-highc-rand_records_20260218103326.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-high35c-rand_records_20260220012209.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-high35c-rand_records_20260216072942.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-medc-rand_records_20260302182741.pkl
data/results/gpt-4.1-mini/kubric-2cue-mark_depth-order-higherc-rand_records_20260221064538.pkl
data/results/gpt-4.1-mini/kubric-2cue-mark_depth-order-higherc-rand_records_20260215125211.pkl
data/results/gpt-4.1-mini/kubric-2cue-aug_depth-order-highc-rand_records_20260217005619.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-highc-rand_records_20260223144950.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-mark_depth-order-higherc-rand_records_20260205162130.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-highc-rand_records_20260201164154.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-medc-rand_records_20260304000052.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-high35c-rand_records_20260224070949.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-medc-rand_records_20260301090751.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-aug_depth-order-high35c-rand_records_20260203165111.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-mark_depth-order-higherc-rand_records_20260225005540.pkl
data/results/InternVL2_5-4B/kubric-2cue-aug_depth-order-high35c-rand_records_20260228023302.pkl
data/results/InternVL2_5-4B/kubric-2cue-aug_depth-order-highc-rand_records_20260226011534.pkl
data/results/InternVL2_5-4B/kubric-2cue-mark_depth-order-higherc-rand_records_20260302032850.pkl
data/results/VILA1.5-3b/kubric-2cue-aug_depth-order-high35c-rand_records_20260225180958.pkl
data/results/VILA1.5-3b/kubric-2cue-mark_depth-order-higherc-rand_records_20260226012234.pkl
data/results/VILA1.5-3b/kubric-2cue-aug_depth-order-medc-rand_records_20260309234726.pkl
data/results/VILA1.5-3b/kubric-2cue-aug_depth-order-highc-rand_records_20260225111900.pkl""").splitlines()
len(CUE_INTERACTION_RESULT_PATHS)

In [ ]:
!find data/results/ -wholename "*fsoc*" -name "*-2cue*depth-order*"

In [ ]:
FSOC_RESULT_PATHS = ("""data/results/llava-v1.5-7b/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303210900.pkl
data/results/google/paligemma2-3b-mix-224/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303211650.pkl
data/results/cambrian-phi3-3b/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303214938.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303173432.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303224911.pkl
data/results/blip2-flan-t5-xl/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303185633.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303203802.pkl
data/results/gemini-2.5-flash-lite/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303202802.pkl
data/results/kosmos-2-patch14-224/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303194856.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303195326.pkl
data/results/Phi-3.5-vision-instruct/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303224358.pkl
data/results/gpt-4.1-mini/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303183231.pkl
data/results/gpt-4.1-mini/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303190616.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303182942.pkl
data/results/DeepSeek-VL-7B-chat/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303180825.pkl
data/results/InternVL2_5-4B/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303190725.pkl
data/results/VILA1.5-3b/kubric-2cue-fsoc-mark_depth-order-higherc-rand_records_20260303200915.pkl""").splitlines()
len(FSOC_RESULT_PATHS)  # 12 reg + 5 ICL

## kb-no-lp

In [ ]:
!find data/results/ -name "*-0lp*depth-order*"

In [ ]:
NOLP_RESULT_PATHS = ("""data/results/llava-v1.5-7b/kubric-0lp-mark_depth-order-higherc-rand_records_20260305164818.pkl
data/results/google/paligemma2-3b-mix-224/kubric-0lp-mark_depth-order-higherc-rand_records_20260305172118.pkl
data/results/cambrian-phi3-3b/kubric-0lp-mark_depth-order-higherc-rand_records_20260305003237.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0lp-aug_depth-order-medc-rand_records_20260323153040.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0lp-mark_depth-order-higherc-rand_records_20260224111317.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0lp-aug_depth-order-highc-rand_records_20260224075704.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0lp-aug_depth-order-high35c-rand_records_20260224094346.pkl
data/results/blip2-flan-t5-xl/kubric-0lp-mark_depth-order-higherc-rand_records_20260227153029.pkl
data/results/blip2-flan-t5-xl/kubric-0lp-aug_depth-order-highc-rand_records_20260227121954.pkl
data/results/blip2-flan-t5-xl/kubric-0lp-aug_depth-order-high35c-rand_records_20260227152739.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-medc-rand_records_20260312015525.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-high35c-rand_records_20260304023557.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-mark_depth-order-higherc-rand_records_20260304025953.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-highc-rand_records_20260304033809.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-mark_depth-order-higherc-rand_records_20260304045012.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-medc-rand_records_20260312015504.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-high35c-rand_records_20260304042737.pkl
data/results/gemini-2.5-flash-lite/kubric-0lp-aug_depth-order-highc-rand_records_20260304021012.pkl
data/results/kosmos-2-patch14-224/kubric-0lp-aug_depth-order-highc-rand_records_20260304033146.pkl
data/results/kosmos-2-patch14-224/kubric-0lp-mark_depth-order-higherc-rand_records_20260304034435.pkl
data/results/kosmos-2-patch14-224/kubric-0lp-aug_depth-order-high35c-rand_records_20260304033847.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-mark_depth-order-higherc-rand_records_20260223133655.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-high35c-rand_records_20260222163607.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-medc-rand_records_20260316163901.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-highc-rand_records_20260222143406.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-highc-rand_records_20260223073353.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-medc-rand_records_20260316204215.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-aug_depth-order-high35c-rand_records_20260223085745.pkl
data/results/Phi-3.5-vision-instruct/kubric-0lp-mark_depth-order-higherc-rand_records_20260223004539.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-highc-rand_records_20260225005142.pkl
data/results/gpt-4.1-mini/kubric-0lp-mark_depth-order-higherc-rand_records_20260226034800.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-medc-rand_records_20260303144057.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-high35c-rand_records_20260225111644.pkl
data/results/gpt-4.1-mini/kubric-0lp-mark_depth-order-higherc-rand_records_20260225121026.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-medc-rand_records_20260303154028.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-high35c-rand_records_20260226011450.pkl
data/results/gpt-4.1-mini/kubric-0lp-aug_depth-order-highc-rand_records_20260225180740.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-highc-rand_records_20260225181853.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-high35c-rand_records_20260226012147.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-highc-rand_records_20260208191935.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-mark_depth-order-higherc-rand_records_20260226034941.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-medc-rand_records_20260321200850.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-high35c-rand_records_20260208180113.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-mark_depth-order-higherc-rand_records_20260208160421.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0lp-aug_depth-order-medc-rand_records_20260319152802.pkl
data/results/InternVL2_5-4B/kubric-0lp-mark_depth-order-higherc-rand_records_20260304061532.pkl
data/results/VILA1.5-3b/kubric-0lp-mark_depth-order-higherc-rand_records_20260226052104.pkl
data/results/VILA1.5-3b/kubric-0lp-aug_depth-order-high35c-rand_records_20260226051136.pkl
data/results/VILA1.5-3b/kubric-0lp-aug_depth-order-highc-rand_records_20260226050309.pkl""").splitlines()
len(NOLP_RESULT_PATHS)

## kb-0cue

In [ ]:
!find data/results/ -name "*-0cue*depth-order*"

In [ ]:
NEG_RESULT_PATHS = ("""data/results/llava-v1.5-7b/kubric-0cue-mark_depth-order-higherc-rand_records_20260305172820.pkl
data/results/google/paligemma2-3b-mix-224/kubric-0cue-mark_depth-order-higherc-rand_records_20260305173426.pkl
data/results/cambrian-phi3-3b/kubric-0cue-mark_depth-order-higherc-rand_records_20260305133140.pkl
data/results/cambrian-phi3-3b/kubric-0cue-aug_depth-order-highc-rand_records_20260303140124.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0cue-aug_depth-order-highc-rand_records_20260214082456.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0cue-aug_depth-order-high35c-rand_records_20260214235944.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/kubric-0cue-mark_depth-order-higherc-rand_records_20260215044952.pkl
data/results/blip2-flan-t5-xl/kubric-0cue-mark_depth-order-higherc-rand_records_20260308193138.pkl
data/results/blip2-flan-t5-xl/kubric-0cue-aug_depth-order-highc-rand_records_20260227153856.pkl
data/results/gemini-2.5-flash-lite/kubric-0cue-mark_depth-order-higherc-rand_records_20260305220249.pkl
data/results/kosmos-2-patch14-224/kubric-0cue-mark_depth-order-higherc-rand_records_20260305185449.pkl
data/results/Phi-3.5-vision-instruct/kubric-0cue-aug_depth-order-highc-rand_records_20260223154108.pkl
data/results/Phi-3.5-vision-instruct/kubric-0cue-aug_depth-order-high35c-rand_records_20260224005128.pkl
data/results/Phi-3.5-vision-instruct/kubric-0cue-mark_depth-order-higherc-rand_records_20260224032624.pkl
data/results/gpt-4.1-mini/kubric-0cue-aug_depth-order-highc-rand_records_20260226071834.pkl
data/results/gpt-4.1-mini/kubric-0cue-mark_depth-order-higherc-rand_records_20260226120929.pkl
data/results/gpt-4.1-mini/kubric-0cue-aug_depth-order-medc-rand_records_20260303162906.pkl
data/results/gpt-4.1-mini/kubric-0cue-aug_depth-order-high35c-rand_records_20260226093035.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0cue-aug_depth-order-high35c-rand_records_20260226070134.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0cue-mark_depth-order-higherc-rand_records_20260226093507.pkl
data/results/DeepSeek-VL-7B-chat/kubric-0cue-aug_depth-order-highc-rand_records_20260226050152.pkl
data/results/InternVL2_5-4B/kubric-0cue-mark_depth-order-higherc-rand_records_20260305191702.pkl
data/results/VILA1.5-3b/kubric-0cue-mark_depth-order-higherc-rand_records_20260226095440.pkl
data/results/VILA1.5-3b/kubric-0cue-aug_depth-order-highc-rand_records_20260226071749.pkl
data/results/VILA1.5-3b/kubric-0cue-aug_depth-order-high35c-rand_records_20260226094016.pkl""").splitlines()
len(NEG_RESULT_PATHS)

## real-012cue

In [ ]:
!find data/results/ -wholename "*bbox-ref*" -name "*o3d-real*depth-order*"

In [ ]:
O3DR_RESULT_PATHS = ("""data/results/llava-v1.5-7b/o3d-real-mark_depth-order-higherc_records_20250619034436.pkl
data/results/llava-v1.5-7b/o3d-real-aug_depth-order-full-v2_records_20250619033218.pkl
data/results/google/paligemma2-3b-mix-224/o3d-real-aug_depth-order-full-v2_records_20250619151247.pkl
data/results/google/paligemma2-3b-mix-224/o3d-real-mark_depth-order-higherc_records_20250619154136.pkl
data/results/cambrian-phi3-3b/o3d-real-aug_depth-order-full-v2_records_20250619223052.pkl
data/results/cambrian-phi3-3b/o3d-real-mark_depth-order-higherc_records_20250620001628.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3d-real-aug_depth-order-full-v2_records_20250620002959.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3d-real-mark_depth-order-higherc_records_20250620015314.pkl
data/results/blip2-flan-t5-xl/o3d-real-mark_depth-order-higherc_records_20250617211357.pkl
data/results/blip2-flan-t5-xl/o3d-real-aug_depth-order-full-v2_records_20250617211208.pkl
data/results/gemini-2.5-flash-lite/o3d-real-mark_depth-order-higherc-rand_records_20260304055735.pkl
data/results/kosmos-2-patch14-224/o3d-real-mark_depth-order-higherc_records_20250620150113.pkl
data/results/kosmos-2-patch14-224/o3d-real-aug_depth-order-full-v2_records_20250620145322.pkl
data/results/Phi-3.5-vision-instruct/o3d-real-aug_depth-order-full-v2_records_20250619154516.pkl
data/results/Phi-3.5-vision-instruct/o3d-real-mark_depth-order-higherc_records_20250619161724.pkl
data/results/gpt-4.1-mini/o3d-real-mark_depth-order-higherc_records_20250618210907.pkl
data/results/gpt-4.1-mini/o3d-real-aug_depth-order-full-v2_records_20250618200524.pkl
data/results/DeepSeek-VL-7B-chat/o3d-real-mark_depth-order-higherc_records_20250618181329.pkl
data/results/DeepSeek-VL-7B-chat/o3d-real-aug_depth-order-full-v2_records_20250618165709.pkl
data/results/InternVL2_5-4B/o3d-real-aug_depth-order-full-v2_records_20250619022816.pkl
data/results/InternVL2_5-4B/o3d-real-mark_depth-order-higherc_records_20250619032301.pkl
data/results/VILA1.5-3b/o3d-real-aug_depth-order-full-v2_records_20250619165419.pkl
data/results/VILA1.5-3b/o3d-real-mark_depth-order-higherc_records_20250619171745.pkl""").splitlines()
len(O3DR_RESULT_PATHS)

## real-012cue-cropped

In [ ]:
!find data/results/ ! -wholename "*bbox-ref*" -name "*o3dcf-real*depth-order*"

In [ ]:
O3DCFR_RESULT_PATHS = ("""data/results/llava-v1.5-7b/o3dcf-real-aug_depth-order-high35c-lr_records_20250619033729.pkl
data/results/llava-v1.5-7b/o3dcf-real-mark_depth-order-higherc_records_20250619034100.pkl
data/results/google/paligemma2-3b-mix-224/o3dcf-real-mark_depth-order-higherc_records_20250619152304.pkl
data/results/google/paligemma2-3b-mix-224/o3dcf-real-aug_depth-order-high35c-lr_records_20250619151822.pkl
data/results/cambrian-phi3-3b/o3dcf-real-mark_depth-order-higherc_records_20250619235206.pkl
data/results/cambrian-phi3-3b/o3dcf-real-aug_depth-order-high35c-lr_records_20250619232157.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3dcf-real-aug_depth-order-high35c-lr_records_20250620010432.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3dcf-real-mark_depth-order-higherc_records_20250620013536.pkl
data/results/blip2-flan-t5-xl/o3dcf-real-aug_depth-order-high35c-lr_records_20250617211542.pkl
data/results/blip2-flan-t5-xl/o3dcf-real-mark_depth-order-higherc_records_20250617211441.pkl
data/results/gemini-2.5-flash-lite/o3dcf-real-mark_depth-order-higherc-rand_records_20260304055116.pkl
data/results/kosmos-2-patch14-224/o3dcf-real-mark_depth-order-higherc_records_20250620150013.pkl
data/results/kosmos-2-patch14-224/o3dcf-real-aug_depth-order-high35c-lr_records_20250620145853.pkl
data/results/Phi-3.5-vision-instruct/o3dcf-real-mark_depth-order-higherc_records_20250619160921.pkl
data/results/Phi-3.5-vision-instruct/o3dcf-real-aug_depth-order-high35c-lr_records_20250619155552.pkl
data/results/gpt-4.1-mini/o3dcf-real-aug_depth-order-high35c-lr_records_20250618204016.pkl
data/results/gpt-4.1-mini/o3dcf-real-mark_depth-order-higherc_records_20250618205222.pkl
data/results/DeepSeek-VL-7B-chat/o3dcf-real-mark_depth-order-higherc_records_20250618174209.pkl
data/results/DeepSeek-VL-7B-chat/o3dcf-real-aug_depth-order-high35c-lr_records_20250618171503.pkl
data/results/InternVL2_5-4B/o3dcf-real-mark_depth-order-higherc_records_20250619030739.pkl
data/results/InternVL2_5-4B/o3dcf-real-aug_depth-order-high35c-lr_records_20250619025509.pkl
data/results/VILA1.5-3b/o3dcf-real-aug_depth-order-high35c-lr_records_20250619170956.pkl
data/results/VILA1.5-3b/o3dcf-real-mark_depth-order-higherc_records_20250619171417.pkl""").splitlines()
len(O3DCFR_RESULT_PATHS)

## real-mcue

In [ ]:
!find data/results/ -name "*o3-dorder*"

In [ ]:
O3DO_RESULT_PATHS = ("""data/results/llava-v1.5-7b/o3-dorder-mark_depth-order-higherc-rand_records_20260305172336.pkl
data/results/llava-v1.5-7b/o3-dorder_depth-order-medc-rand_records_20260305171925.pkl
data/results/google/paligemma2-3b-mix-224/o3-dorder-mark_depth-order-higherc-rand_records_20260305181013.pkl
data/results/google/paligemma2-3b-mix-224/o3-dorder_depth-order-medc-rand_records_20260305181338.pkl
data/results/cambrian-phi3-3b/o3-dorder-mark_depth-order-higherc-rand_records_20260305020156.pkl
data/results/cambrian-phi3-3b/o3-dorder_depth-order-medc-rand_records_20260305193957.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3-dorder_depth-order-medc-rand_records_20260305183551.pkl
data/results/Qwen2-VL-7B-Instruct-GPTQ-Int4/o3-dorder-mark_depth-order-higherc-rand_records_20260305190350.pkl
data/results/blip2-flan-t5-xl/o3-dorder_depth-order-medc-rand_records_20260305181649.pkl
data/results/blip2-flan-t5-xl/o3-dorder-mark_depth-order-higherc-rand_records_20260305182719.pkl
data/results/gemini-2.5-flash-lite/o3-dorder-mark_depth-order-higherc-rand_records_20260304053610.pkl
data/results/gemini-2.5-flash-lite/o3-dorder_depth-order-medc-rand_records_20260305173122.pkl
data/results/gemini-2.5-flash-lite/o3-dorder_depth-order-medc-rand_records_20260304051731.pkl
data/results/gemini-2.5-flash-lite/o3-dorder-mark_depth-order-higherc-rand_records_20260305174925.pkl
data/results/kosmos-2-patch14-224/o3-dorder-mark_depth-order-higherc-rand_records_20260305185133.pkl
data/results/kosmos-2-patch14-224/o3-dorder_depth-order-medc-rand_records_20260305183423.pkl
data/results/Phi-3.5-vision-instruct/o3-dorder_depth-order-medc-rand_records_20260223151851.pkl
data/results/Phi-3.5-vision-instruct/o3-dorder-mark_depth-order-higherc-rand_records_20260223152606.pkl
data/results/Phi-3.5-vision-instruct/o3-dorder_depth-order-medc-rand_records_20260223144619.pkl
data/results/Phi-3.5-vision-instruct/o3-dorder-mark_depth-order-higherc-rand_records_20260223140624.pkl
data/results/gpt-4.1-mini/o3-dorder-mark_depth-order-higherc-rand_records_20260226051026.pkl
data/results/gpt-4.1-mini/o3-dorder_depth-order-medc-rand_records_20260226050020.pkl
data/results/gpt-4.1-mini/o3-dorder-mark_depth-order-higherc-rand_records_20260226051944.pkl
data/results/gpt-4.1-mini/o3-dorder_depth-order-medc-rand_records_20260226070044.pkl
data/results/DeepSeek-VL-7B-chat/o3-dorder_depth-order-medc-rand_records_20260226121054.pkl
data/results/DeepSeek-VL-7B-chat/o3-dorder-mark_depth-order-higherc-rand_records_20260226125221.pkl
data/results/DeepSeek-VL-7B-chat/o3-dorder-mark_depth-order-higherc-rand_records_20260222021331.pkl
data/results/DeepSeek-VL-7B-chat/o3-dorder_depth-order-medc-rand_records_20260222015017.pkl
data/results/InternVL2_5-4B/o3-dorder-mark_depth-order-higherc-rand_records_20260305185056.pkl
data/results/InternVL2_5-4B/o3-dorder_depth-order-medc-rand_records_20260305182615.pkl
data/results/VILA1.5-3b/o3-dorder-mark_depth-order-higherc-rand_records_20260226121141.pkl
data/results/VILA1.5-3b/o3-dorder_depth-order-medc-rand_records_20260226122613.pkl""").splitlines()
len(O3DO_RESULT_PATHS)

# concat

In [ ]:
import tqdm

## kb-0/1/2cue

In [ ]:
# takes 2 mins to load
dfs1 = []
for path in tqdm.tqdm(SINGLE_CUE_RESULT_PATHS):
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_kb1m_res_gt,
    ).pipe(_df_add_global_info)
    dfs1.append(df)
cue1 = (
    pd.concat(dfs1, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)
print("loaded 1cue")

In [ ]:
dfs2 = []
for path in tqdm.tqdm(FSOC_RESULT_PATHS+CUE_INTERACTION_RESULT_PATHS):
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_kb2m_res_gt,
    ).pipe(_df_add_global_info)
    dfs2.append(df)
cue2 = (
    pd.concat(dfs2, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)
print("loaded 2cue")

cue2.cues.nunique()

In [ ]:
dfs0 = []
for path in tqdm.tqdm(NEG_RESULT_PATHS):
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_kb0m_res_gt,
    ).pipe(_df_add_global_info)
    dfs0.append(df)
    
cue0 = (
    pd.concat(dfs0, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)

cue1.shape, cue2.shape, cue0.shape

In [ ]:
dfs0lp = []
for path in tqdm.tqdm(NOLP_RESULT_PATHS):
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_kb0lpm_res_gt,
    ).pipe(_df_add_global_info)
    dfs0lp.append(df)
    
cue0lp = (
    pd.concat(dfs0lp, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)
cue0lp.shape

## real-012cue/-cropped

In [ ]:
# see if can concat o3d-r & o3dcf-r; YES, but w/ two metric cols

print("loading 012cue")
dfs_r = []
for path in O3DR_RESULT_PATHS:
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherBinaryEvaluator(), core.TextResult,
                                 core.TextGroundTruth("farther"), gt_src_getter=None),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_o3dr_res_gt,
    ).pipe(_df_add_global_info)
    dfs_r.append(df)
o3dr = (
    pd.concat(dfs_r, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)

print("loading 012cue-cropped")
dfs_cfr = []
for path in O3DCFR_RESULT_PATHS:
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(core.CloserFartherO3dcfrEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_o3drcf_res_gt,
    ).pipe(_df_add_global_info)
    dfs_cfr.append(df)
o3dcfr = (
    pd.concat(dfs_cfr, axis=0, ignore_index=True)
        .pipe(_df_add_row_info)
)

o3dr_ = pd.concat([o3dr, o3dcfr], axis=0, ignore_index=True)
o3dr.shape, o3dcfr.shape, o3dr_.shape

In [ ]:
o3dr_.filter(like="Closer").columns

## o3-dorder

In [ ]:
class CloserFartherO3DorderEvaluator(core.BaseEvaluator):
    def evaluate(self, result: core.TextResult, gt: core.DataframeRowGT):
        actual_answer = result.text.lower()

        gt_dict = gt.load()
        odd_position = gt_dict["odd_position"]
        if odd_position == "near":
            expected_answers = core.CloserFartherFilenameEvaluator.EXPECTED_NEAR_ANSWERS
        elif odd_position == "far":
            expected_answers = core.CloserFartherFilenameEvaluator.EXPECTED_FAR_ANSWERS
        else:
            raise ValueError(f'Unsupported odd_position value: {odd_position}')

        return any(expected in actual_answer for expected in expected_answers)

In [ ]:
dfs = []
for path in O3DO_RESULT_PATHS:
    df = run_analysis_pipeline(
        eval_defs=[
            EvaluationDefinition(CloserFartherO3DorderEvaluator(), core.TextResult,
                                 core.DataframeRowGT),
        ],
        result_pickle_path=path,
        filter_res=_parse_qa,
        merge_res_gt=_merge_o3do_res_gt,
    ).pipe(_df_add_global_info)
    dfs.append(df)

o3_do = (
    pd.concat(dfs, axis=0, ignore_index=True)
)
o3_do.shape

## fillna for `icl`

In [ ]:
cue1 = cue1.assign(icl=lambda df: df.icl.fillna(False))
cue2 = cue2.assign(icl=lambda df: df.icl.fillna(False))
cue0lp = cue0lp.assign(icl=lambda df: df.icl.fillna(False))
o3_do = o3_do.assign(icl=lambda df: df.icl.fillna(False))

# DepthAnyV2
**Depth maps can be generated with HF model: Depth-Anything-V2-Small-hf**

### DAny eval classes

In [ ]:
class _DepthAnythingFilepathResult(core.TextResult):
    IMAGE_SET: expt.BaseImageSet = None

    DEPTH_ANYTHING_DIR_NAME = "danything_maps"

    @classmethod
    def from_record(cls, record: dict):
        """Get depthanything output filepath from image info in `record`."""
        img_path = cls.IMAGE_SET.get_image_path(record["full_image_name"], ignore_nonexist=True)
        # get depthanything file path from image path
        depth_path = str(util.format_subdir(img_path, cls.DEPTH_ANYTHING_DIR_NAME, sibling=True)) + '.npy'

        text = depth_path
        return cls(text=text, truncated=False)

class SingleCueAugDepthAnythingFilepathResult(_DepthAnythingFilepathResult):
    IMAGE_SET = expt.KubricOneCueAugmentedImageSet()

class CueInteractionAugDepthAnythingFilepathResult(_DepthAnythingFilepathResult):
    IMAGE_SET = expt.KubricTwoCueAugmentedImageSet()

class NegativeAugDepthAnythingFilepathResult(_DepthAnythingFilepathResult):
    IMAGE_SET = expt.KubricNoCueAugmentedImageSet()


class O3DRealAugDepthAnythingFilepathResults(_DepthAnythingFilepathResult):
    IMAGE_SET = expt.O3DepthRealAugmentedImageSet()
    
class O3DCFRealAugDepthAnythingFilepathResults(_DepthAnythingFilepathResult):
    IMAGE_SET = expt.O3DepthCFRealAugmentedImageSet()

In [ ]:
import random

class DAnyCloserFartherEvaluator(core.BaseEvaluator):
    SUBDIR = "augmented/"
    TARG_MASK_DIR = "targ_labels/"
    DIST_MASK_DIR = "dist_labels/"
    SEGMT_SUBDIR = "orig/"

    def evaluate(self, result: _DepthAnythingFilepathResult, gt: core.DataframeRowGT):
        gt_dict = gt.load()

        expected_answers = ['closer'] if gt_dict['odd_position'] == 'near' else ['farther', 'further']

        # targ/dist labels
        # ASSUMING same shape as depth map
        img_path = result.IMAGE_SET.get_image_path(gt_dict["image_name"], ignore_nonexist=True)
        targ_path = self._get_mask_path(img_path, self.TARG_MASK_DIR)
        dist_path = self._get_mask_path(img_path, self.DIST_MASK_DIR)
        # targ_path = str(img_path).replace("images/", "targ_segmts/").replace(self.SUBDIR, self.SEGMT_SUBDIR)
        # dist_path = str(img_path).replace("images/", "dist_segmts/").replace(self.SUBDIR, self.SEGMT_SUBDIR)

        # depth map
        dany_filepath = result.text
        
        # calc targ/dist median depth
        targ_med_depth = self._get_depth_stats(dany_filepath, targ_path, np_stats=np.median)
        dist_med_depth = self._get_depth_stats(dany_filepath, dist_path, np_stats=np.median)

        if targ_med_depth > dist_med_depth:  # DepthAnything raw value: greater is closer
            actual_answer = 'closer'
        elif targ_med_depth < dist_med_depth:
            actual_answer = 'farther'
        else:
            actual_answer = random.choice(['closer', 'farther'])

        return any(expected == actual_answer for expected in expected_answers)


    @functools.lru_cache()
    def _get_depth_stats(self, dmap_path, mask_path, np_stats=None):
        dmap = np.load(dmap_path)

        o3mask = plt.imread(mask_path)
        if len(o3mask.shape) == 3:
            o3mask = o3mask[:, :, :3].max(axis=-1)

        masked_dmap = self._apply_mask(o3mask, dmap)
        return np_stats(masked_dmap)

    def _get_mask_path(self, img_path, mask_dirname):
        return str(img_path).replace("images/", mask_dirname).replace(self.SUBDIR, self.SEGMT_SUBDIR)

    def _apply_mask(self, o3mask, dmap):
        bin_mask = self._bin_mask(o3mask)
        mask_d_arr = dmap[np.nonzero(bin_mask)]
        return mask_d_arr

    @staticmethod
    def _bin_mask(o3mask):
        thresh = 0.5 if o3mask.max() <= 1 else 128
        return np.where(o3mask > thresh, np.ones_like(o3mask), np.zeros_like(o3mask))


### DAny, kb-0cue

In [ ]:
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/kubric_scenes/no_cue/images/danything_maps/
danykb0 = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCloserFartherEvaluator(), NegativeAugDepthAnythingFilepathResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=NEG_RESULT_PATHS[0],  # as a placeholder
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]),
    merge_res_gt=_merge_kb0m_res_gt,
).assign(cue_strength_str=lambda df: df.cue_strength.astype(str).str.slice(0, 4))
danykb0.shape

In [ ]:
danykb0[lambda df: df.odd_position.isin({"near", "far"})].DAnyCloserFarther.mean()

### DAny kb-1cue

In [ ]:
%%time
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/kubric_scenes/one_cue/images/danything_maps/
danykb1 = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCloserFartherEvaluator(), SingleCueAugDepthAnythingFilepathResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=SINGLE_CUE_RESULT_PATHS[0],
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]).sample(frac=.5),
    merge_res_gt=_merge_kb1m_res_gt,
).assign(cue_strength_str=lambda df: df.cue_strength.astype(str).str.slice(0, 4))
danykb1.shape

In [ ]:
danykb1.DAnyCloserFarther.mean()

In [ ]:
(
    danykb1
        .assign(cues=lambda df: df.cues.apply(lambda l: eval(l)[0]))
        .groupby("cues").DAnyCloserFarther.mean()
        [["HP", "OC", "RS", "TG", "SA", "LS", "FO", "FS"]]
        .round(4)
        .pipe(lambda s: " & ".join(s.astype(str).to_numpy()))
)

### Dany kb-2cue

In [ ]:
%%time
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/kubric_scenes/two_cue/images/danything_maps/
danykb2 = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCloserFartherEvaluator(), CueInteractionAugDepthAnythingFilepathResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=CUE_INTERACTION_RESULT_PATHS[0],  # as placeholder
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]).sample(frac=.5),
    merge_res_gt=_merge_kb2m_res_gt,
).assign(cue_strength_str=lambda df: df.cue_strength.astype(str).str.slice(0, 4))
danykb2.shape

In [ ]:
%%time
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/kubric_scenes/two_cue/images/danything_maps/
danykb2fsoc = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCloserFartherEvaluator(), CueInteractionAugDepthAnythingFilepathResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=FSOC_RESULT_PATHS[0],
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]),
    merge_res_gt=_merge_kb2m_res_gt,
).assign(cue_strength_str=lambda df: df.cue_strength.astype(str).str.slice(0, 4))
danykb2fsoc.shape

In [ ]:
danykb2_ = pd.concat([danykb2, danykb2fsoc])
danykb2_.shape

In [ ]:
danykb2_.DAnyCloserFarther.mean()

In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

ax = (
    pd.concat([danykb1, danykb2_])
        .assign(cues=lambda df: df.cues.map(eval).map(sorted).map(tuple))
        .groupby("cues")
        .DAnyCloserFarther.mean()
        .reset_index()
        .assign(cue1=lambda df: df.cues.apply(lambda t: sorted(t)[0]),
                cue2=lambda df: df.cues.apply(lambda t: sorted(t)[1] if len(t) == 2 else t[0]))
        .pivot(index="cue2", columns="cue1", values="DAnyCloserFarther").T
        .pipe(
            (sns.heatmap, "data"),
            cmap="coolwarm", vmin=0, vmax=1,
            annot=True, annot_kws=dict(fontsize=12), cbar=False, square=True,
            # xticklabels=False,
        )
)
ax.set(xlabel=None, ylabel="DepthAnyV2 accuracy")
ax.yaxis.tick_right()
ax.xaxis.tick_top()
ax.set_ylabel("DepthAnyV2 accuracy", labelpad=15)
ax.yaxis.label_position = "right"
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.savefig("dany-1-2-cue.png", transparent=True)

### DAny real-012

In [ ]:
class DAnyCamReadyCloserFartherEvaluator(DAnyCloserFartherEvaluator):
    TARG_MASK_DIR = "targ_segmts/"
    DIST_MASK_DIR = "dist_segmts/"
    SEGMT_SUBDIR = "augmented/"

    def _get_mask_path(self, img_path, mask_dirname):
        mask_path = super()._get_mask_path(img_path, mask_dirname)
        return mask_path.replace(".jpg", ".png")

In [ ]:
%%time
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/o3d_real/images/danything_maps/
dany_o3dr = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCamReadyCloserFartherEvaluator(), O3DRealAugDepthAnythingFilepathResults,
                             core.DataframeRowGT),
    ],
    result_pickle_path=O3DR_RESULT_PATHS[0], # as a placeholder
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]),
    merge_res_gt=_merge_o3dr_res_gt,
)
dany_o3dr.shape

In [ ]:
dany_o3dr.DAnyCamReadyCloserFarther.mean()

In [ ]:
# need depth maps estimated by depthanything saved as .npy files in <data_dir>/o3dcf_real/images/danything_maps/
dany_o3dcfr = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(DAnyCamReadyCloserFartherEvaluator(), O3DCFRealAugDepthAnythingFilepathResults,
                             core.DataframeRowGT),
    ],
    result_pickle_path=O3DCFR_RESULT_PATHS[0],  # as placeholder
    filter_res=lambda df: df.drop_duplicates("image_name").drop(columns=["ques"]),
    merge_res_gt=_merge_o3drcf_res_gt,
)
dany_o3dcfr.shape

In [ ]:
dany_o3dcfr.DAnyCamReadyCloserFarther.mean()

In [ ]:
(
    pd.concat([dany_o3dr, dany_o3dcfr])
        .assign(n_cues=lambda df: df.cues.apply(eval).apply(len))
        .groupby("n_cues").DAnyCamReadyCloserFarther.mean()
)  # real 0, 1, & 2-cue, DA mean

### overall

In [ ]:
( # depthany 1-cue 2-cue real sim overall
    pd.concat([danykb1, danykb2_], axis=0)
        .DAnyCloserFarther.mean().round(4)
)/2 + 0.9859/2

In [ ]:
pd.concat([dany_o3dr, dany_o3dcfr]).loc[lambda df: df.cues.apply(eval).apply(len).ge(1)].DAnyCamReadyCloserFarther.mean()
# real 1 & 2 cue, dany

In [ ]:
(
    pd.concat([dany_o3dr, dany_o3dcfr])
        .assign(n_cues=lambda df: df.cues.apply(eval).apply(len))
        .groupby("n_cues").DAnyCamReadyCloserFarther.mean()
)  # real 0, 1, & 2-cue, DA mean

In [ ]:
danykb1.loc[lambda df: df.cue_strength == 1].DAnyCloserFarther.mean()
# 1 cue, reg str

In [ ]:
danykb2_.DAnyCloserFarther.mean()

In [ ]:
(
    pd.concat([dany_o3dr, dany_o3dcfr])
        .assign(n_cues=lambda df: df.cues.apply(eval).apply(len))
        .loc[lambda df: df.n_cues == 1]
        .DAnyCamReadyCloserFarther.mean()
)/2 + danykb1.loc[lambda df: df.cue_strength == 1].DAnyCloserFarther.mean()/2
# 1 cue, real&sim

In [ ]:
(
    pd.concat([dany_o3dr, dany_o3dcfr])
        .assign(n_cues=lambda df: df.cues.apply(eval).apply(len))
        .loc[lambda df: df.n_cues == 2]
        .DAnyCamReadyCloserFarther.mean()
)/2 + danykb2_.DAnyCloserFarther.mean()/2
# 2 cue, real&sim

# sanity checks
## for `kb12`

In [ ]:
kb12 = (
    pd.concat([cue1, cue2], axis=0, ignore_index=True)
)
kb12.shape

In [ ]:
kb12.cues.nunique()
# 8 (1-cue) + 28 (2-cue) = 36;

In [ ]:
(kb12
     .groupby(["model_display_name", "icl"]).size().unstack(level=0, fill_value=0)
     .style.background_gradient()
)
# not all models support ICL/CoT

In [ ]:
(kb12
     .groupby(["model_display_name", "cues"]).size().unstack(level=0, fill_value=0)
     # .sum().mean(axis=1)
     .style.background_gradient()
)
# FS has fewer examples because of few object pairs

In [ ]:
(kb12
     .loc[lambda df: df.icl.eq(False) & df.ques_clarity.ge(3)]
     .groupby(["model_display_name", "is_marked"]).size().unstack(level=0).style.background_gradient())

In [ ]:
(kb12
     .loc[lambda df: df.icl.eq(False) & df.ques_clarity.ge(3)]
     .groupby(["model_display_name", "cue_strength_str"]).size().unstack(level=0, fill_value=0)
     .style.background_gradient())

In [ ]:
(kb12[non_icl_filter]
     .loc[lambda df: df.ques_clarity == 4]
     .groupby(["model_display_name", "env_id"]).size().unstack(level=0, fill_value=0)
     # .sum()
     # .mean(axis=1)
     .style.background_gradient()
)

In [ ]:
kb12.obj_id.value_counts()
# (src, small obj name) for FS cues, e.g. ('gso', 'CoQ10_BjTLbuRVt1t')

## for `cue0lp` (kb-no-lp)

In [ ]:
(cue0lp[non_icl_filter]
     .loc[lambda df: df.ques_clarity.eq(4)]
     .groupby(["model_display_name"]).size()
)

## for `cue0`

In [ ]:
(cue0[non_icl_filter]
     .loc[lambda df: df.ques_clarity.eq(4)]
     .groupby(["model_display_name"]).size()
)

## for `o3dr` & `o3dcfr`

In [ ]:
o3dr.shape, o3dr[aug_mark_filter].shape

In [ ]:
(
    o3dr[aug_mark_filter]
        .assign(n_cues=lambda df: df.cues.str.len())
        .groupby(["ques_clarity", "n_cues"]).size().unstack()
        .sum(axis=1)
)

In [ ]:
( # o3dr imset cue distribution
    o3dr
        .drop_duplicates("image_name")
        .assign(cues=lambda df: df.cues.apply(sorted))
        .loc[lambda df: df.cues.str.len() > 0]
        .assign(cue1=lambda df: df.cues.apply(lambda t: t[0]),
                cue2=lambda df: df.cues.apply(lambda t: t[0] if len(t) == 1 else t[1]))
        .groupby(["cue1", "cue2"]).size().unstack(fill_value=0)
        .style.background_gradient(axis=None)
)

In [ ]:
o3dr[["image_name", "cues"]].loc[lambda df: df.cues == ()].drop_duplicates()

## for `o3_do`

In [ ]:
o3_do.image_name.nunique()

In [ ]:
o3_do.ques_clarity.value_counts()

In [ ]:
(
    o3_do[non_icl_filter]
        .groupby(["model_display_name", "odd_position"]).size().unstack(0)
        .style.background_gradient()
)

# Figure, performance summary

### quick metrics

In [ ]:
kb12.icl.value_counts()

In [ ]:
kb12.groupby("ques_clarity").CloserFartherFilename.mean()

In [ ]:
kb12.groupby("model_display_name").CloserFartherFilename.mean()

In [ ]:
(kb12[non_icl_filter]
     .loc[lambda df: df.ques_clarity.ge(3)]
     .groupby(["model_display_name", "ques_clarity"]).CloserFartherFilename.mean().unstack(level=0)
     .style.background_gradient(vmin=0, vmax=1, cmap='coolwarm'))

In [ ]:
(kb12
     .groupby(["model_display_name", "icl"]).CloserFartherFilename.mean().unstack(level=0)
     .style.background_gradient(vmin=0, vmax=1, cmap='coolwarm'))

##### gpt, dig deeper

In [ ]:
(kb12.loc[lambda df: df.model_display_name == "gpt-4.1-mini"]
     .groupby(["cues", "icl"]).CloserFartherFilename.mean().unstack(level=-1)
     .sort_values(True)
     .style.background_gradient(vmin=0, vmax=1, cmap='coolwarm')
)
# best three: RS, HP, FS

### D metric

In [ ]:
d_metric_sim = (
    kb12
        .loc[non_icl_filter]
        .groupby(["model_display_name", "ques_clarity"]).CloserFartherFilename.mean().unstack()
        .loc[:, 4].to_frame()  # clarity=4
        .rename(columns={4: "y_depth"})
)
d_metric_sim.pipe(_df_sbg)

#### real, cl4, cue12m;

In [ ]:
o3d_overall = (
    pd.concat([o3_do[non_icl_filter],
               o3dcfr[aug_mark_filter],
              ], axis=0, ignore_index=True)
        .assign(CloserFartherCombined=lambda df: df.pipe(_df_combine_metrics, prefix="CloserFarther"))
)
o3d_overall.shape

##### some model performed bettern on sim

In [ ]:
d_metric_r = (
    o3d_overall
        .groupby(["model_display_name", "ques_clarity"]).CloserFartherCombined.mean().unstack()
        .loc[:, 4]  # cl=4
        .to_frame().rename(columns={4: "y_depth"})
)
# d_metric_r.style.background_gradient(vmin=0, vmax=1, cmap='coolwarm')

(
    pd.concat([d_metric_sim['y_depth'].rename("sim"),
               d_metric_r['y_depth'].rename("real")], axis=1)
        .diff(axis=1).style.background_gradient(vmin=-.5, vmax=.5, cmap='coolwarm')
)

#### s+r

In [ ]:
d_snr = (
    pd.concat([kb12, o3_do, o3dcfr], axis=0, ignore_index=True)
        .loc[non_icl_filter]
        .assign(CFCombined=lambda df: df.pipe(_df_combine_metrics, prefix='CloserFarther'))
)
d_snr.shape

In [ ]:
d_metric = (
    d_snr[aug_mark_filter]
        .groupby(["model_display_name", "ques_clarity"]).CFCombined.mean().unstack()
        .loc[:, 4]  # cl=4
        .to_frame().rename(columns={4: "y_depth"})
)
d_metric.style.background_gradient(vmin=0, vmax=1, cmap='coolwarm')

d_metric_df = (
    pd.concat([d_metric_sim['y_depth'].rename("sim"),
               d_metric_r['y_depth'].rename("real"),
               d_metric['y_depth'].rename("s+r")], axis=1)
        .assign(sr_mean=lambda df: df.sim.add(df.real).div(2),
                r_minus_s=lambda df: df.real - df.sim)  # treat both side as a sample
)
d_metric_df.pipe(_df_sbg)

In [ ]:
d_metric_df[["sim", "real", "r_minus_s"]].mean()

### D&L | std(acc) & L metric
comparing vision influence and language influence

In [ ]:
QID_MAP = {  # key words -> qid
'closer to or farther away': 'Q9',
 'behind or in front of': 'Q8',
 'feel farther or nearer': 'Q7',
 'positioned before or after': 'Q6',
 'come before or after': 'Q5',
 'at the rear of': 'Q4',
 'Relative to the camera': 'Q3',
 'True or False': 'Q2',
 'positioned farther from or closer to': 'Q1'    
}

def _to_qid(formatted_query):
    for kw, qid in QID_MAP.items():
        if kw in formatted_query:
            return qid
    raise ValueError(f"Can't map QID for: {formatted_query}")

def _add_qid(df):
    return (
        df.ques.str.split("A.", regex=False).apply(lambda l: l[0])
            .map(_to_qid)
    )

(
    kb12[kb12.model_display_name == 'deepseek-vl-7b-chat']
        .assign(qid=lambda df: _add_qid(df))
        .qid.value_counts()
)

In [ ]:
(
    kb12[reg_strength_filter]
        .loc[non_icl_filter]
        .loc[lambda df: df.odd_position.eq("near")]
        .loc[lambda df: df.ques_clarity.ge(3)]
        .assign(qtags_t=lambda df: df.qtags.map(tuple))
        .groupby(["qtags_t", "ques_clarity"]).size().unstack()
)  # 14 x 3 = 42

In [ ]:
kb_dl = (
    kb12[reg_strength_filter]
        .loc[non_icl_filter]
        .loc[lambda df: df.odd_position.eq("near")]
        .loc[lambda df: df.ques_clarity.ge(3)]
        .assign(qid=lambda df: _add_qid(df))
        .assign(qv_group=lambda df: df.ques_clarity.astype(str).str.cat(df.qtags.astype(str)))
)

In [ ]:
kb_dl.cues.value_counts(dropna=False).shape, kb_dl.groupby('qv_group').size().shape
# similar numbers of groups

In [ ]:
kb_dl.groupby(["model_display_name", "qv_group"]).CloserFartherFilename.size().unstack(0, fill_value=0)

#### L metric
1 - SDGM

In [ ]:
l3_metric_sim = 1 - (
    kb_dl.groupby(["model_display_name", "qv_group"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby("model_display_name").CloserFartherFilename.std()
        .rename("lang")
)
l3_metric_sim

#### fig, V-L influence

In [ ]:
snsb, snso = sns.color_palette('deep')[:2]

In [ ]:
vlm_order = d_metric_df.sr_mean.sort_values(ascending=False).index
vlm_order

In [ ]:
model_formal_name_map = {
    'qwen2-vl-7b-instruct-gptq-int4': "Qwen2-VL",
    'paligemma2-3b-mix-224': "PaliGemma2",
    'internvl2_5-4b': "InternVL2.5", 
    'phi-3.5-vision-instruct': "Phi3.5", 
    'cambrian-phi3-3b': "Cambrian",
    'deepseek-vl-7b-chat': "DeepSeek-VL", 
    'llava-v1.5-7b': "LLaVA1.5", 
    'gpt-4.1-mini': "GPT4.1-mini", 
    'gemini-2.5-flash-lite': "Gemini2.5",
    'vila1.5-3b': "VILA1.5",
    'blip2-flan-t5-xl': "BLIP2", 
    'kosmos-2-patch14-224': "Kosmos2",
    'depthanythingv2': 'DepthAnythingV2',
}

model_short_name_map = {
    'qwen2-vl-7b-instruct-gptq-int4': "Qw",
    'paligemma2-3b-mix-224': "Pg",
    'internvl2_5-4b': "Ir", 
    'phi-3.5-vision-instruct': "Ph", 
    'cambrian-phi3-3b': "Cb",
    'deepseek-vl-7b-chat': "Ds", 
    'llava-v1.5-7b': "Lv", 
    'gpt-4.1-mini': "Gp", 
    'gemini-2.5-flash-lite': "Ge",
    'vila1.5-3b': "Vi",
    'blip2-flan-t5-xl': "Bp", 
    'kosmos-2-patch14-224': "Km",
    'depthanythingv2': 'Da',
}

cue_name_map = {
    ('HP',): "Height-in-Plane",
    ('LS',): "Light-and-Shadow",
    ('OC',): "Occlusion",
    ('RS',): "Relative Size",
    ('SA',): "Saturation",
    ('TG',): "Texture Gradient",
    ('FO',): "Focusness",
    ('FS',): "Familiar Size",
    ('LP',): "Linear Perspective",
}

marker_names = "ov^spP*hXdD8>"

m_order = pd.Series(vlm_order.tolist() + ["depthanythingv2"]).map(model_short_name_map).to_list()
marker_map = dict(zip(m_order, marker_names))
marker_map

In [ ]:
vl_influ = pd.concat([
    kb_dl.groupby(["model_display_name", "cues"], as_index=False).CloserFartherFilename.mean().groupby("model_display_name").CloserFartherFilename.std()
        .rename("vision"),
    kb_dl.groupby(["model_display_name", "qv_group"], as_index=False).CloserFartherFilename.mean().groupby("model_display_name").CloserFartherFilename.std()
        .rename("language"),
], axis=1).reset_index()
vl_influ

In [ ]:
# mpl.rcParams['figure.dpi'] = 150

In [ ]:
with sns.axes_style("whitegrid"):
    g = (
        vl_influ
            .assign(vision=lambda df: df.vision.mul(-1),
                    model_formal_name=lambda df: df.model_display_name.map(model_formal_name_map))
            # .assign(model_formal_name=lambda df: df.model_formal_name.apply(lambda s: " "*((12-len(s))//2) + s))
            .drop(columns="model_display_name")
            .sort_values("language")
            .rename(columns={"vision": "Vision Influence", "language": "Language influence"})
            .melt(id_vars="model_formal_name", var_name="modality", value_name="sigma")
            .pipe((sns.catplot, "data"), kind='bar',
                  col='modality',
                  hue='modality', palette=[snso, snsb],
                  x='sigma', y="model_formal_name",
                  sharex=False,
                  aspect=4/3, height=3,
                  facet_kws=dict(gridspec_kws=dict(wspace=0.39)),
                  legend=False,
                 )
                  
    )
    g.set_ylabels("")
    g.set_xlabels(r"$\sigma(\mathrm{Acc})$")
    g.set_titles(col_template="{col_name}")

axl = g.axes[0,0]
axr = g.axes[0,1]

axl.set(xlim=(-0.5, 0))
axl.yaxis.tick_right()
axl.tick_params(length=0)
axl.spines[['left', 'top']].set_visible(False)
axl.spines[['bottom']].set_visible(False)

axr.set(xlim=(0, 0.5))
# axr.spines[['right', 'top']].set_visible(False)
axr.spines[['bottom']].set_visible(False)

axr_ticklabels = axr.get_xticklabels()
axl.set_xticklabels(axr_ticklabels[::-1])

plt.tight_layout()
# plt.savefig('vl_influ.pdf', bbox_inches='tight')

### fig scatterplot, summary

In [ ]:
depthany_d_metric = 0.84745
y_metric = d_metric_df.sr_mean
x_metric = l3_metric_sim

In [ ]:
xy_metric = pd.concat([x_metric, y_metric], axis=1)
xy_sim_metric = pd.concat([l3_metric_sim, d_metric_df.sim], axis=1)

In [ ]:
xy_sim_metric.loc["qwen2-vl-7b-instruct-gptq-int4"].values

In [ ]:
# w/ L3 as x-axis
mpl.rcParams['figure.dpi'] = 100
# mpl.rcParams['figure.dpi'] = 300

xlim_max = 0.9

__, ax = plt.subplots(figsize=(5,5))
for (x, y), l, m in zip(xy_metric.loc[vlm_order].values, vlm_order.map(model_formal_name_map), marker_names):
    ax.scatter(x, y, marker=m, label=l, alpha=1)

ax.axhline(depthany_d_metric, color='gray', ls='--', lw=.5)  # depth-anything-v2-small-hf
ax.axhline(0.5, color='gray', ls='-', lw=.5, zorder=-1)

ax.set(
    xlim=(0.4, xlim_max),
    # xlim=(0.75, 1),
    ylim=(0.4, .9),
    xlabel="Language consistency", ylabel="Depth ordering accuracy")
ax.legend(bbox_to_anchor=(-0.01, 0.98), loc="upper left")
ax.annotate(" DepthAnything", xy=(xlim_max, depthany_d_metric), va='center', c='gray')
ax.annotate(" Random guess", xy=(xlim_max, 0.5), va='center', c='gray')
ax.spines[['right', 'top']].set_visible(False)

from matplotlib.patches import FancyArrowPatch

gap_arrow_top_coord = xy_metric.loc["qwen2-vl-7b-instruct-gptq-int4"].values
gap_arrow_bottom_coord = (xy_metric.loc["qwen2-vl-7b-instruct-gptq-int4"].values[0], depthany_d_metric)
ax.annotate(
    '',
    xy=gap_arrow_top_coord,
    xytext=gap_arrow_bottom_coord,
    arrowprops=dict(
    arrowstyle='<->',
    color='xkcd:dark red',
    lw=1,
    shrinkA=10, shrinkB=10)
)
ax.annotate(
    ("Large gap\nb/w baseline\nand VLMs\n"),
    xy=(xy_metric.loc["qwen2-vl-7b-instruct-gptq-int4"].values[0]+ 0.01,
        (xy_metric.loc["qwen2-vl-7b-instruct-gptq-int4"].values[1]+depthany_d_metric)/2),
    ha="left", va="center",
    color='xkcd:dark red',
)

spread_arrow_y = 0.435
spread_arrow_l_coord = (xy_metric.loc['vila1.5-3b'].values[0], spread_arrow_y)
spread_arrow_r_coord = (xy_metric.loc['internvl2_5-4b'].values[0], spread_arrow_y)
ax.annotate(
    '',
    xy=spread_arrow_l_coord,
    xytext=spread_arrow_r_coord,
    arrowprops=dict(
    arrowstyle='<->',
    color='xkcd:cadet blue',
    lw=1,
    shrinkA=0, shrinkB=0)
)
ax.annotate(
    "Wider spread of language consistency",
    xy=((xy_metric.loc['vila1.5-3b'].values[0]+xy_metric.loc['internvl2_5-4b'].values[0])/2,
        spread_arrow_y-0.005),
    ha="center", va="top",
    color='xkcd:cadet blue',
)

ax.annotate(
    '',
    xy=(.78, .78),
    xytext=(.87, .87),
    arrowprops=dict(
        arrowstyle='<-',
        color='xkcd:sage',
        lw=1, ls=':',
        shrinkA=0, shrinkB=0
    )
)
ax.annotate(
    "Better",
    xy=(.835, .855),
    rotation=45,
    ha="center", va="center",
   color='xkcd:sage',
)

# plt.tight_layout()
# plt.savefig('d_l_scores.pdf', bbox_inches='tight')

- [x] could a model stuck at a constant output and have high L score?
    - sure, it is consistently under-performing

# D | 0/1/2-cue; vlm-level

In [ ]:
def float_df_to_latex_tab(df, cols):
    return (
        df
        .applymap(lambda n: f"{n:.4f}")
        .assign(model_formal_name=lambda df: df.index.to_series().map(model_formal_name_map))
        [["model_formal_name"]+cols]
        .apply(lambda row: " & ".join(row) + r" \\", axis=1)
    )

In [ ]:
cue012_vlm_sim = (
    pd.concat([cue0, kb12], axis=0, ignore_index=True)
        .loc[non_icl_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .assign(n_cues=lambda df: df.cues.str.len())
        .groupby(["model_display_name", "n_cues"]).CloserFartherFilename.mean().unstack().astype(float).round(4)
        # .mean()
        # .diff(axis=1)  # NOTE. few improved: LV, QW  & depthany
        # .pipe(_df_sbg)
)

In [ ]:
cue012_vlm_sim.mean().round(4)

In [ ]:
cue1_vlm = (
    cue1[aug_mark_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .assign(cue_name=lambda df: df.cues.apply(lambda seq: seq[0]))
        .groupby(["model_display_name", "cue_name"]).CloserFartherFilename.mean().unstack().astype(float).round(4)
)

In [ ]:
(
    kb12
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .assign(n_cues=lambda df: df.cues.str.len())
        .groupby(["model_display_name", "n_cues"]).CloserFartherFilename.mean().unstack()
        .pipe(_df_sbg)
)

In [ ]:
(
    kb12
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .assign(n_cues=lambda df: df.cues.str.len())
        .groupby(["model_display_name", "n_cues"]).CloserFartherFilename.mean().unstack()
)

### baseline: real scene

In [ ]:
o3dcfr.columns

In [ ]:
cue012_vlm_real = (
    pd.concat([o3dcfr.loc[lambda df: df.cues.str.len() == 0], o3dcfr[reg_strength_filter]],
              axis=0, ignore_index=True)
        .loc[aug_mark_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .assign(n_cues=lambda df: df.cues.str.len())
        # .n_cues.value_counts()  # 2: 462; 1: 374
        .groupby(["model_display_name", "n_cues"]).CloserFartherO3dcfr.mean().unstack().round(4)
)

In [ ]:
(
    o3dcfr
        .loc[aug_mark_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .assign(n_cues=lambda df: df.cues.str.len())
        # .n_cues.value_counts()  # 2: 462; 1: 374
        .groupby(["model_display_name", "n_cues"]).CloserFartherO3dcfr.mean().unstack()
        .pipe(_df_sbg)
)

### concat axis=1

In [ ]:
cue1_vlm.index, cue012_vlm_sim.index, cue012_vlm_real.index

In [ ]:
(
    pd.concat([
        cue1_vlm,
        cue012_vlm_sim.add_prefix("sim_cue"),
        cue012_vlm_real.add_prefix("real_cue"),
    ], axis=1)
        .pipe(float_df_to_latex_tab, cols=["HP", "OC", "RS", "TG", "SA", "LS", "FO", "FS", "sim_cue0", "sim_cue1", "sim_cue2",
                                           "real_cue0", "real_cue1", "real_cue2"])
        .apply(print)
)

In [ ]:
(
    pd.concat([
        cue1_vlm,
        cue012_vlm_sim.add_prefix("sim_cue"),
        cue012_vlm_real.add_prefix("real_cue"),
    ], axis=1)
        .mean().to_frame().T
        [["HP", "OC", "RS", "TG", "SA", "LS", "FO", "FS", "sim_cue0", "sim_cue1", "sim_cue2","real_cue0", "real_cue1", "real_cue2"]]
        .apply(lambda row: " & ".join(row.round(4).astype(str))+r" \\", axis=1)
        .apply(print)
)

## LP vs no LP

In [ ]:
(
    kb12.loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.env_id == "basic"]
        .loc[non_icl_filter]
        .loc[lambda df: df.cues.apply(lambda t: "HP" in t and "FS" not in t)]
        .cues.value_counts()
)

In [ ]:
(
    cue0lp.loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.env_id == "basic"]
        .loc[non_icl_filter]
        .cues.value_counts()
)

In [ ]:
(
    kb12.loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.env_id == "basic"]
        .loc[non_icl_filter]
        .loc[lambda df: df.cues.apply(lambda t: "HP" in t and "FS" not in t)]
        .CloserFartherFilename.mean()
) - (
    cue0lp.loc[lambda df: df.ques_clarity == 4]
        .loc[non_icl_filter]
        .CloserFartherFilename.mean()
)

In [ ]:
(
    cue0lp.loc[lambda df: df.ques_clarity == 4]
        .loc[non_icl_filter]
        .CloserFartherFilename.mean()
)

In [ ]:
kb12_wlp = (
    kb12.loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.env_id == "basic"]
        .loc[non_icl_filter]
        .loc[lambda df: df.cues.apply(lambda t: "HP" in t and "FS" not in t)]
        .assign(cues=lambda df: df.cues.map(sorted).map(tuple))
        .groupby("cues").CloserFartherFilename.mean()
        .rename("with_lp")
)

In [ ]:
kb12_nolp = (
    cue0lp.loc[lambda df: df.ques_clarity == 4]
        .loc[non_icl_filter]
        .assign(cues=lambda df: df.cues.map(sorted).map(tuple))
        .groupby("cues").CloserFartherFilename.mean()
        .rename("no_lp")
)

In [ ]:
kb12_wlp - kb12_nolp

In [ ]:
kb12_lp_nolp = pd.DataFrame(
    {"nolp": kb12_nolp, "lp": kb12_wlp, "diff": kb12_wlp - kb12_nolp}
)
kb12_lp_nolp.shape

In [ ]:
kb12_lp_nolp.index

In [ ]:
rows = [('HP',),('FO', 'HP'), ('HP', 'LS'), ('HP', 'OC'), ('HP', 'RS'),
       ('HP', 'SA'), ('HP', 'TG')]

In [ ]:
" & ".join(str(c) for c in rows)

In [ ]:
pd.set_option('display.max_colwidth', 200)
(
    kb12_lp_nolp.loc[rows,:].T.round(4)
        .applymap(lambda n: f"{n:.4f}")
        .apply(lambda row: " & ".join(row) + r" \\", axis=1)
)

In [ ]:
kb12_wlp_vlm = (
    kb12.loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.env_id == "basic"]
        .loc[non_icl_filter]
        .loc[lambda df: df.cues.apply(lambda t: "HP" in t and "FS" not in t)]
        .assign(cues=lambda df: df.cues.map(sorted).map(tuple))
        .groupby(["model_display_name", "cues"]).CloserFartherFilename.mean()
        .unstack(0)[['cambrian-phi3-3b', 'deepseek-vl-7b-chat',
       'gemini-2.5-flash-lite', 'gpt-4.1-mini', 'internvl2_5-4b',
       'phi-3.5-vision-instruct', 'qwen2-vl-7b-instruct-gptq-int4',
       ]]
)

In [ ]:
kb12_nolp_vlm = (
    cue0lp.loc[lambda df: df.ques_clarity == 4]
        .loc[non_icl_filter]
        .assign(cues=lambda df: df.cues.map(sorted).map(tuple))
        .groupby(["model_display_name", "cues"]).CloserFartherFilename.mean()
        .unstack(0)[['cambrian-phi3-3b', 'deepseek-vl-7b-chat',
       'gemini-2.5-flash-lite', 'gpt-4.1-mini', 'internvl2_5-4b',
       'phi-3.5-vision-instruct', 'qwen2-vl-7b-instruct-gptq-int4',
       ]]
)

In [ ]:
(
    (kb12_wlp_vlm - kb12_nolp_vlm)
        [['cambrian-phi3-3b', 'deepseek-vl-7b-chat',
       'gemini-2.5-flash-lite', 'gpt-4.1-mini', 'internvl2_5-4b',
       'phi-3.5-vision-instruct', 'qwen2-vl-7b-instruct-gptq-int4',
       ]]
        .mean(axis=1)
        # .pipe(_df_bg_coolwarm, vmin=-1)
)

In [ ]:
kb12_lp_nolp_vlm = pd.DataFrame(
    {"nolp": kb12_nolp_vlm.mean(axis=1), "lp": kb12_wlp_vlm.mean(axis=1), "diff": (kb12_wlp_vlm - kb12_nolp_vlm).mean(axis=1)}
)
kb12_lp_nolp_vlm.shape

In [ ]:
kb12_lp_nolp_vlm

In [ ]:
kb12_lp_nolp_vlm.loc[rows,:].T

In [ ]:
(
    kb12_lp_nolp_vlm.loc[rows,:].T.round(4)
        .applymap(lambda n: f"{n:.4f}")
        .apply(lambda row: " & ".join(row) + r" \\", axis=1)
)

# L | clarity

In [ ]:
kb12_cl_mean = (
    kb12
        .loc[non_icl_filter]
        .loc[lambda df: df.ques_clarity.ge(3)]
        .groupby(["model_display_name", "ques_clarity"])
        .CloserFartherFilename.mean().unstack()
        .mean()
)
print(" & ".join(kb12_cl_mean.round(4).astype(str)))
kb12_cl_mean

#### refline: km bbox @0.4671

In [ ]:
KM_RESULT_PATH = pathlib.Path("data/results/kosmos-2-patch14-224/bbox-ref")

bbox ref results
```
kubric-1cue-aug_depth-order-high35c-rand_records_20260309172352.pkl
kubric-2cue-aug_depth-order-high35c-rand_records_20260309201139.pkl
kubric-2cue-fsoc-aug_depth-order-high35c-rand_records_20260309231516.pkl
kubric-0lp-aug_depth-order-high35c-rand_records_20260309231651.pkl
```

In [ ]:
kmbb_kb1_path = KM_RESULT_PATH / "kubric-1cue-aug_depth-order-high35c-rand_records_20260309172352.pkl"
kmbb_kb1 = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=kmbb_kb1_path,
    filter_res=_parse_qa,
    merge_res_gt=_merge_kb1m_res_gt,
).pipe(_df_add_global_info).pipe(_df_add_row_info)
kmbb_kb1.shape

kmbb_kb2_path = KM_RESULT_PATH / "kubric-2cue-aug_depth-order-high35c-rand_records_20260309201139.pkl"
kmbb_kb2 = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=kmbb_kb2_path,
    filter_res=_parse_qa,
    merge_res_gt=_merge_kb2m_res_gt,
).pipe(_df_add_global_info).pipe(_df_add_row_info)
kmbb_kb2.shape

kmbb_kb2fsoc_path = KM_RESULT_PATH / "kubric-2cue-fsoc-aug_depth-order-high35c-rand_records_20260309231516.pkl"
kmbb_kb2fsoc = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=kmbb_kb2fsoc_path,
    filter_res=_parse_qa,
    merge_res_gt=_merge_kb2m_res_gt,
).pipe(_df_add_global_info).pipe(_df_add_row_info)
kmbb_kb2fsoc.shape

kmbb_kb0lp_path = KM_RESULT_PATH / "kubric-0lp-aug_depth-order-high35c-rand_records_20260309231651.pkl"
kmbb_kb0lp = run_analysis_pipeline(
    eval_defs=[
        EvaluationDefinition(core.CloserFartherFilenameEvaluator(), core.TextResult,
                             core.DataframeRowGT),
    ],
    result_pickle_path=kmbb_kb0lp_path,
    filter_res=_parse_qa,
    merge_res_gt=_merge_kb0lpm_res_gt,
).pipe(_df_add_global_info).pipe(_df_add_row_info)
kmbb_kb0lp.shape

kmbb_kb12 = (
    pd.concat([kmbb_kb1, kmbb_kb2, kmbb_kb2fsoc, kmbb_kb0lp], ignore_index=True)
        .assign(cues=lambda df: df.cues.apply(sorted).apply(tuple))
)
print(kmbb_kb12.shape)
kmbb_kb12.cues.nunique()

In [ ]:
kmbb_kb12.CloserFartherFilename.mean().round(4)

In [ ]:
kmbb_kb12.groupby("odd_position").CloserFartherFilename.mean()

#### baseline: real

In [ ]:
(
    o3dcfr # 1 & 2 cues
        .loc[lambda df: df.cues.apply(len).ge(1)]
        .groupby(["model_display_name", "ques_clarity"])
        .CloserFartherO3dcfr.mean().unstack().mean()
)

#### (VLM vs ques clarity, real)

In [ ]:
vlm_qcl_real = (
    o3dcfr
        .loc[lambda df: df.cues.apply(len).ge(1)]   # 1 & 2 cues
        .groupby(["model_display_name", "ques_clarity"])
        .CloserFartherO3dcfr.mean().unstack()
)

#### (VLM vs ques clarity, sim)

In [ ]:
_txt = """BLIP2 & 0.4917 & 0.5032 & 0.4995 & 0.4987 &- & 0.4845 \\
    Cambrian & 0.4874 & 0.5025 & 0.4733 & 0.5208 &-  & 0.5155 \\
    DeepSeek-VL & 0.4807 & 0.5133 & 0.5219 & 0.5438 &- & 0.5052 \\
    Gemini2.5 & 0.4327 & 0.4076 & 0.4979 & 0.5128 &- & 0.5876 \\
    GPT4.1-mini & 0.4925 & 0.5347 & 0.5559 & 0.5207 &- & 0.5876 \\
    InternVL2.5 & 0.5016 & 0.5076 & 0.5232 & 0.5168 &- & 0.5670 \\
    Kosmos2 & 0.3878 & 0.3979 & 0.4305 & 0.4347 & 0.4671 & 0.5361 \\
    LLaVA1.5 & 	0.5005 & 0.4989 & 0.4992 & 0.5063 &- & 0.5206 \\
    PaliGemma2 & 0.4807 & 0.5043 & 0.5108 & 0.4755 &- & 0.5876 \\
    Phi3.5 & 0.4953 & 0.5035 & 0.5138 & 0.5234 &- & 0.5464 \\
    Qwen2-VL & 0.5134 & 0.5176 & 0.5343 & 0.5225 &- & 0.5567 \\
    VILA1.5 & 0.5035 & 0.5015 & 0.5097 & 0.5052 &- & 0.4639 \\
""".splitlines()
sum([float(l.split("&")[1].strip()) for l in _txt]) / len([float(l.split("&")[1].strip()) for l in _txt])

In [ ]:
vlm_qcl_sim = (
    kb12
        .loc[non_icl_filter]
        .loc[lambda df: df.ques_clarity.ge(3)]
        .groupby(["model_display_name", "ques_clarity"])
        .CloserFartherFilename.mean().unstack()
)

#### concat

In [ ]:
vlm_qcl_sim.columns, vlm_qcl_real.columns

In [ ]:
(
    pd.concat([vlm_qcl_sim.add_prefix("sim_cl"),
               vlm_qcl_real.add_prefix("real_cl")], axis=1)
        .pipe(float_df_to_latex_tab, cols=[
            # 'sim_cl2.0',
            'sim_cl3.0', 'sim_cl3.5', 'sim_cl4.0',
            # 'real_cl3.5',
            'real_cl4.0',
        ])
        .apply(print)
)

In [ ]:
(
    pd.concat([vlm_qcl_sim.add_prefix("sim_cl"),
               vlm_qcl_real.add_prefix("real_cl")], axis=1)
        .mean().to_frame().T
        .drop(columns="real_cl3.5")
        .apply(lambda row: " & ".join(row.round(4).astype(str)) + r"\\", axis=1)
        .apply(print)
)

# D | ICL & CoT
- sim for now
- clarity >= 3, i.e. hi, hi35, hier

In [ ]:
ds_icl_fil = lambda df: df.model_display_name.eq('deepseek-vl-7b-chat') & df.ques_clarity.ge(3)
ph_icl_fil = lambda df: df.model_display_name.eq('phi-3.5-vision-instruct') & df.ques_clarity.ge(3)
qw_icl_fil = lambda df: df.model_display_name.eq('qwen2-vl-7b-instruct-gptq-int4') & df.ques_clarity.ge(3)
gp_icl_fil = lambda df: df.model_display_name.eq('gpt-4.1-mini') & df.ques_clarity.ge(3)
gm_icl_fil = lambda df: df.model_display_name.eq('gemini-2.5-flash-lite') & df.ques_clarity.ge(3)
kb12_ = kb12.assign(n_cues=lambda df: df.cues.apply(len))

kb12_icl = (
    pd.concat([
        kb12_.loc[ds_icl_fil],
        kb12_.loc[ph_icl_fil],
        kb12_.loc[qw_icl_fil],
        kb12_.loc[gp_icl_fil],
        kb12_.loc[gm_icl_fil]
    ])
)
kb12_icl.icl.value_counts()

In [ ]:
(kb12_icl.groupby(["model_display_name", "icl"]).CloserFartherFilename.mean().unstack()
     .rename(columns={True: "icl", False: "no_icl"})
     .assign(impr=lambda df: df.icl - df.no_icl)
     # .pipe(_df_sbg, vmin=-1)
     .pipe(float_df_to_latex_tab, ["no_icl", "icl", "impr"])
)

In [ ]:
(kb12_icl
     .loc[lambda df: df.ques_clarity == 4]
     .groupby(["model_display_name", "icl"]).CloserFartherFilename.mean().unstack()
     .pipe(_df_sbg)
)

In [ ]:
(kb12_icl
     .groupby(["model_display_name", "ques_clarity", "icl"]).CloserFartherFilename.mean().unstack([-2,-1])
     .pipe(_df_sbg)
)

In [ ]:
(kb12_icl.groupby(["model_display_name", "icl"]).CloserFartherFilename.mean().unstack(0)
     .diff()
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
cue_cols = ['HP', 'OC', 'RS', 'TG', 'SA', 'LS', 'FO', 'FS']

print(" & ".join(cue_cols))

(kb12_icl.loc[lambda df: df.n_cues == 1]
     .groupby(["cues", "icl"]).CloserFartherFilename.mean().unstack(-1)
     .reset_index().assign(cues=lambda df: df.cues.apply(lambda t: t[0])).set_index("cues")
     .rename(columns={True: "icl", False: "no_icl"})
     .assign(impr=lambda df: df.icl - df.no_icl)
     # .pipe(_df_bg_coolwarm)
     .T
     .round(4)
     [cue_cols].apply(lambda r: " & ".join([r.name]+r.astype(str).tolist()), axis=1)
     
)

In [ ]:
(kb12_icl.loc[lambda df: df.n_cues == 1]
     .groupby(["cues", "icl"]).CloserFartherFilename.mean().unstack(0)
     .diff()
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
(kb12_icl.groupby(["model_display_name", "cues", "icl"]).CloserFartherFilename.mean().unstack()
     .diff(axis=1).iloc[:, [-1]]
     .unstack()
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
(kb12_icl.groupby(["model_display_name", "cues", "icl"]).CloserFartherFilename.mean().unstack()
     .diff(axis=1).iloc[:, [-1]]
     .unstack()
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
(kb12_icl.loc[lambda df: df.n_cues == 1]
     .groupby(["model_display_name", "cues", "icl"]).CloserFartherFilename.mean().unstack()
     .diff(axis=1).iloc[:, [-1]]
     .unstack()
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
(kb12_icl.groupby(["cues", "icl"]).CloserFartherFilename.mean().unstack()
     .diff(axis=1).iloc[:, [-1]]
     .sort_values(True)
     .pipe(_df_sbg, vmin=-1)
)

In [ ]:
print(kb12.loc[lambda df: df.icl & df.cues.apply(len).eq(1)].demos.sample().to_numpy()[0])

# D | basic results
supporting and explaning D.1

## 0 cue.
sim + real
- o3dcfr used for its bal near/far

In [ ]:
cue0_ = (
    pd.concat([
        cue0[lambda df: df.odd_position.isin({"near", "far"})],
        o3dcfr[lambda df: df.cues.str.len().eq(0)],
    ], axis=0, ignore_index=True)
        .assign(CFMetric=lambda df: df.pipe(_df_combine_metrics, prefix="CloserFarther"))
)
cue0_.shape

(
    cue0_
        .loc[lambda df: df.ques_clarity == 4]
        .CFMetric.mean()
)

## 1 cue.
sim + real
- o3dcfr used for its bal near/far

In [ ]:
(
    cue1[non_icl_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .CloserFartherFilename.mean()
)

In [ ]:
cue1_ = (
    pd.concat([
        cue1[non_icl_filter],
        o3dcfr[lambda df: df.cues.str.len().eq(1)],
    ], axis=0, ignore_index=True)
        .assign(CFMetric=lambda df: df.pipe(_df_combine_metrics, prefix="CloserFarther"))
)

(
    cue1_
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .CFMetric.mean()
)

# 1-cue (sim + real)

## 1 cue. by strengths

In [ ]:
set_mpl_dpi(100)
# set_mpl_dpi()

g = (
    cue1_[aug_mark_filter]
        .loc[non_icl_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .loc[lambda df: df.cue_strength.isin({1,2})]
        .groupby(["cue_strength", "model_display_name", "env_id", "cues"])
        # .size()
        .CFMetric.mean()
        .reset_index()
        .assign(cue_str_str=lambda df: df.cue_strength.astype(str).str.slice(0,3),
                model_formal_name=lambda df: df.model_display_name.map(model_formal_name_map))
        .assign(cue_name=lambda df: df.cues.map(cue_name_map),
                cue_abbr=lambda df: df.cues.apply(lambda t: t[0]))
        .pipe(
            (sns.catplot, "data"), kind='bar',
            y="cue_str_str", x="CFMetric",
            col="cue_abbr", col_order=["HP", "OC", "RS", "TG", "SA", "LS", "FO", "FS"],
            row="model_formal_name",
            height=1.2, aspect=2/1,
        )
)
g.refline(x=0.5)

g.set_ylabels("Cue strength")
g.set_xlabels("Depth ordering acc.")

g.set_titles(col_template="{col_name}", row_template="{row_name}")

# plt.savefig("cue_strengths_vlms.pdf", bbox_inches='tight')

## 2 cue

In [ ]:
(
    cue2[non_icl_filter]
        .assign(CFMetric=lambda df: df.pipe(_df_combine_metrics, prefix="CloserFarther"))
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .CFMetric.mean()
)
# 2-cue (sim);

In [ ]:
cue2_ = (
    pd.concat([
        cue2[non_icl_filter],
        o3dcfr[lambda df: df.cues.str.len().eq(2)],
    ], axis=0, ignore_index=True)
        .assign(CFMetric=lambda df: df.pipe(_df_combine_metrics, prefix="CloserFarther"))
)
cue2_.columns

(
    cue2_
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .CFMetric.mean()
)
# 2-cue (real + sim);

## 2 cue. heatmap (incl 1-cue)

In [ ]:
ax = (
    pd.concat([cue2_, cue1_], axis=0, ignore_index=True)
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .groupby(["cues"])
        # .size()  # ~ 400
        .CFMetric.mean()
        .reset_index()
        .assign(cue1=lambda df: df.cues.apply(lambda t: sorted(t)[0]),
                cue2=lambda df: df.cues.apply(lambda t: sorted(t)[1] if len(t) == 2 else t[0]))
        .pivot(index="cue2", columns="cue1", values="CFMetric")
        .pipe(
            (sns.heatmap, "data"),
            cmap="coolwarm", vmin=0, vmax=1,
            annot=True, annot_kws=dict(fontsize=12),
            cbar=False, square=True,
        )
        
)

ax.set(xlabel="", ylabel="VLM mean accuracy")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# plt.savefig("vlm-1-2-cue.png")

## 1 cue & 2 cue; by models

In [ ]:
depthany_c1_reg_str = 0.7587778316019879
depthany_c2_reg_str = 0.8722813294416721

In [ ]:
group_var = "env_id"

c1m = (
    cue1_[aug_mark_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .loc[reg_strength_filter]
        .groupby(["model_display_name", "env_id"])
        # .size()
        .CFMetric.mean()
        .sort_values(ascending=False)  # so later model ranked by highest metric
        .reset_index()
        .assign(model_short_name=lambda df: df.model_display_name.str.slice(0, 2))
        # .unstack(level=0).std()
)

c2m = (
    cue2_[aug_mark_filter]
        .loc[lambda df: df.ques_clarity == 4]
        .groupby(["model_display_name", "env_id"])
        # .size()
        .CFMetric.mean()
        .reset_index()
        .assign(model_short_name=lambda df: df.model_display_name.str.slice(0, 2))
)

g = (
    pd.concat([c1m.assign(n_cues=1),
               c2m.assign(n_cues=2)], axis=0, ignore_index=True)
        .rename(columns={"n_cues": "# cues"})
        .assign(model_formal_name=lambda df: df.model_display_name.map(model_formal_name_map))
        .pipe(
            (sns.catplot, "data"), kind='bar',
            x="model_formal_name", y="CFMetric",
            col="# cues", color=snsb,
            order=[model_formal_name_map[n] for n in vlm_order],
            aspect=1, height=4,
        )
)

g.set_ylabels("Depth ordering accuracy")
g.set_xlabels("")
g.axes[0,0].axhline(depthany_c1_reg_str, label="DepthAnyV2", color='gray', ls='--')
g.axes[0,1].axhline(depthany_c2_reg_str, label="DepthAnyV2", color='gray', ls='--')
g.refline(y=0.5, label='random', ls="-", lw=.5)
g.axes[0,1].legend()

ax = g.axes[0,0]
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', va='top')
ax.set(title="One cue")
ax = g.axes[0,1]
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', va='top')
ax.set(title="Two cues")


plt.tight_layout()
# plt.savefig('1_2_cues_vlms.pdf', bbox_inches='tight')

# L | lang dim results

## as L.2, modified SDGM; (consistency #2)

In [ ]:
cols_by_qtags = {
    'is_yn': lambda df: df.qtags.apply(lambda t: 'yn' in t),
    'is_ro': lambda df: df.qtags.apply(lambda t: 'ro' in t),
    'do_rel': lambda df: df.qtags.apply(lambda t: [s for s in t if s.startswith("do-")][0]),
    'f_lvl': lambda df: df.qtags.apply(lambda t: [s for s in t if s.startswith("lvl-")][0]),
}

In [ ]:
pvar3_qcl_sim = (
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques_clarity.ge(3)]
        .assign(**cols_by_qtags)
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl", "ques_clarity"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl"]).CloserFartherFilename.std()
        .unstack(level=0).mean()
        .rename("qcl")
)

In [ ]:
pvar3_ynq_sim = (
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques_clarity.ge(3)]
        .assign(**cols_by_qtags)
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl", "ques_clarity"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby(["model_display_name", "is_ro", "do_rel", "f_lvl", "ques_clarity"]).CloserFartherFilename.std()
        .unstack(level=0).mean()
        .rename("ynq")
)

In [ ]:
pvar3_ro_sim = (
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques_clarity.ge(3)]
        .assign(**cols_by_qtags)
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl", "ques_clarity"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby(["model_display_name", "is_yn", "do_rel", "f_lvl", "ques_clarity"]).CloserFartherFilename.std()
        .unstack(level=0).mean()
        .rename("ro")
)

In [ ]:
pvar3_flvl_sim = (
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques_clarity.ge(3)]
        .assign(**cols_by_qtags)
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl", "ques_clarity"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "ques_clarity"]).CloserFartherFilename.std()
        .unstack(level=0).mean()
        .rename("flvl")
)

In [ ]:
pvar3_do_sim = (
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques_clarity.ge(3)]
        .assign(**cols_by_qtags)
        .groupby(["model_display_name", "is_yn", "is_ro", "do_rel", "f_lvl", "ques_clarity"], as_index=False)
        .CloserFartherFilename.mean()
        .groupby(["model_display_name", "is_yn", "is_ro", "f_lvl", "ques_clarity"]).CloserFartherFilename.std()
        .unstack(level=0).mean()
        .rename("do_rel")
)

### new concat

In [ ]:
pvar3_sim = (
    pd.concat([
        pvar3_qcl_sim,
        pvar3_do_sim, pvar3_flvl_sim, pvar3_ynq_sim, pvar3_ro_sim, 
        kb_dl.groupby(["model_display_name", "qv_group"], as_index=False).CloserFartherFilename.mean()
            .groupby("model_display_name").CloserFartherFilename.std().rename("all_qv")
    ], axis=1)
)
(
    pvar3_sim
        .pipe(_df_sbg, cmap="PuBu")
)

In [ ]:
print(" & ".join(pvar3_sim.mean().round(4).astype(str)))
pvar3_sim.mean().round(4).astype(str)

In [ ]:
(
    pvar3_sim
        .assign(model_formal_name=lambda df: df.index.to_series().map(model_formal_name_map))
        .set_index('model_formal_name')
        # could drop qcl as it is not best comparable with the other three
        .apply(lambda row: " & ".join(row.round(4).astype(str)), axis=1)
)

### Y/N influence

In [ ]:
targ_col = "ynq"

In [ ]:
# as ynq is "is behind?", expecting better far acc when ynq=True; so warming from False->True
# - NOTE: DS, PG are exceptions (over correction?)
# - NOTE: DS, GPT, PG (when exclude reverse option)
(
    kb12[aug_mark_filter]
        .loc[lambda df: df.odd_position.eq("far")]
        .assign(**{targ_col: lambda df: df.qtags.apply(lambda t: 'yn' in t)})
        .loc[lambda df: df.qtags.apply(lambda t: 'ioo' not in t)]
        .groupby([targ_col, "model_display_name"])
        .CloserFartherFilename.mean().unstack(level=0)
        # .diff(axis=1)
        # .style.background_gradient(vmin=-1, vmax=1, cmap='coolwarm')
)

In [ ]:
print((
    kb12[non_icl_filter]
        .loc[lambda df: df.odd_position.eq("near") & df.ques.str.contains("True or False")]
).shape)
# two ques template had similar number of records
# NEW: need to drop this because it's hard to eval the influence when now
#     there are two YN questions and one of each (near vs far)

tf_ques_index = (  # ~= "is it near"
    kb12[non_icl_filter]
        .loc[lambda df: df.ques.str.contains("True or False")]
        .index
)

In [ ]:
kb12_ynq = (   
    kb12[non_icl_filter]
        .drop(index=tf_ques_index)
        .loc[lambda df: df.ques_clarity.ge(3)]  # needed to exclude q-c == 2
        .loc[lambda df: df.odd_position.eq("far")]
        .assign(**{targ_col: lambda df: df.qtags.apply(lambda t: 'yn' in t)})
)
kb12_ynq.groupby(["model_display_name", "ques_clarity"]).size().unstack(0)

In [ ]:
# as ynq is "is behind?", expecting better far acc when ynq=True; so warming from False->True
# -  DS are exceptions
vlm_ynq_sim = (
    kb12_ynq
        .groupby([targ_col, "model_display_name"])
        .CloserFartherFilename.mean().unstack(level=0)
)
vlm_ynq_sim.diff(axis=1).style.background_gradient(vmin=-1, vmax=1, cmap='coolwarm')


In [ ]:
(
    vlm_ynq_sim.assign(delta=lambda df: df.diff(axis=1).iloc[:, -1])
        .rename(columns={False: "MC", True: "YN",})
        # .pipe(_df_bg_coolwarm, vmin=-1)
        .pipe(float_df_to_latex_tab, cols=["MC", "YN", "delta"])
        .apply(print)
)

In [ ]:
print(" & ".join(vlm_ynq_sim.assign(delta=lambda df: df.diff(axis=1).iloc[:, -1]).mean().round(4).astype(str)))

vlm_ynq_sim.assign(delta=lambda df: df.diff(axis=1).iloc[:, -1]).mean().round(4).astype(str)

# Supp Info

## q tags and examples

In [ ]:
def _enhance_qtags(row):

    cl_map = {
        2.0: "low",
        3.0: "med",
        3.5: "high",
        4.0: "highest",
    }
    ref_cl_tag = f"cl-{cl_map[row.ques_clarity]}"
    ref_cl_tag = cl_map[row.ques_clarity]

    # if row.ques_clarity <= 3:
    #     ref_desc_tag = 'sal' if 'sal' in row.qtags else 'stand-out'
    # else:
    #     ref_desc_tag = 'n/a'

    if 'yn' in row.qtags:
        query_tag = 'yes-no'
    else:
        query_tag = 'multi-choice'

    inst_tag = 'reversed' if 'ro' in row.qtags else 'normal'

    formal_tag, rel_tag = row.qtags[1], row.qtags[-1]
    formal_tag = {
        'lvl-avg': 'regular',
        'lvl-formal': 'formal',
        'lvl-casual': 'casual',
    }[formal_tag]
    rel_tag = {
        'do-cf': 'close-far',
        'do-rf': 'front-rear',
        'do-ba': 'before-after',
    }[rel_tag]

    tags = (('clarity', ref_cl_tag), ('formality', formal_tag), ('depth', rel_tag), ('query', query_tag), ('option', inst_tag))
    return tags

qt_cnt = (
    cue1[non_icl_filter]
        .loc[lambda df: df.model_display_name == 'qwen2-vl-7b-instruct-gptq-int4']
        .assign(qtags_=lambda df: df.apply(_enhance_qtags, axis=1))
        # .drop_duplicates('qtags_').ques.values
        .groupby('qtags_', as_index=False).size()
)
qt_cnt

In [ ]:
def _format_tag_ques_row(row: pd.Series):
    return rf"{row.tag_dimension} & \texttt{{{row.tag_name}}} & {row.ques} \\"

(
    cue1[non_icl_filter]
        .loc[lambda df: df.model_display_name == 'qwen2-vl-7b-instruct-gptq-int4']
        .assign(qtags_=lambda df: df.apply(_enhance_qtags, axis=1))
        .loc[:, ["qtags_", "ques"]]
        # .assign(ques=lambda df: df.ques.str.split(" A. ").apply(lambda l: l[0]))
        .explode("qtags_")
        .groupby("qtags_").sample(1)
        .assign(tag_dimension=lambda df: df.qtags_.map(lambda t: t[0]),
                tag_name=lambda df: df.qtags_.map(lambda t: t[1]))
        [["tag_dimension", "tag_name", "ques"]]
        .apply(_format_tag_ques_row, axis=1)
        .map(print)
)

# misc

In [ ]:
kb12.ques.sample().values

## kubric assets

In [ ]:
(kb12[(kb12.cues == ('HP',))]
     .loc[aug_mark_filter]
     .loc[reg_strength_filter]
     .loc[lambda df: df.ques_clarity.lt(3)]
     .drop_duplicates("obj_id")
     [["full_image_name", "obj_id"]]
)